# Charging analysis based on trajectory data

In [1]:
import cudf
import cupy as cp
import pandas as pd
import numpy as np
import os
from pathlib import Path
from scipy.spatial import cKDTree
from heapq import heappush, heappop
import pyarrow.parquet as pq
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import logging
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
import pickle
from sklearn.cluster import DBSCAN

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")


In [2]:
# Haversine distance
def haversine_distance(lon1, lat1, lon2, lat2):
    lon1 = cp.asarray(lon1.values if hasattr(lon1, 'values') else lon1)
    lat1 = cp.asarray(lat1.values if hasattr(lat1, 'values') else lat1)
    lon2 = cp.asarray(lon2.values if hasattr(lon2, 'values') else lon2)
    lat2 = cp.asarray(lat2.values if hasattr(lat2, 'values') else lat2)

    lon1_rad = cp.radians(lon1.astype(cp.float64))
    lat1_rad = cp.radians(lat1.astype(cp.float64))
    lon2_rad = cp.radians(lon2.astype(cp.float64))
    lat2_rad = cp.radians(lat2.astype(cp.float64))

    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    a = cp.sin(dlat / 2)**2 + cp.cos(lat1_rad) * cp.cos(lat2_rad) * cp.sin(dlon / 2)**2
    c = 2 * cp.arcsin(cp.sqrt(a))
    R = 6371000
    return c * R


## Parameters


In [47]:
# parameters about identifying charing
R_STATION = 200          # distance to station (m）
T_STAY = 600             # min stay duration (second)
T_GAP = 300              # max time interval (seconds) between consecutive points in range of a station
PASSENGER_THR = 0        # idle

# parameters about energy
CAP_KWH = 80             # battery capacity（kWh）
CONS_KWH_PER_KM = 0.2   # consumption rate（kWh/km）
AVG_SPEED_KMH = 20       # （km/h）, used for lost data estimation
CHG_EFFICIENCY = 1.0     # actual charging percentage

MAX_GAP_SECONDS = 300   #The data loss time (seconds) is 10 minutes, otherwise ignore energy consumption

# file path
TRACKS_DIR = '../data/taxi_trajectories.parquet'  # trajectory dict
STATION_CSV = 'station_information.csv'           # station
PILE_CSV = 'pile_rated_power.csv'                 # piles


# output path
OUTPUT_DIR = 'output_v5'
os.makedirs(OUTPUT_DIR, exist_ok=True)
DIST_CHARGING_EVENTS_FILE = f'{OUTPUT_DIR}/distributed_charging_events.parquet'
CHAR_QUEUE_FILE = f'{OUTPUT_DIR}/char_queue.parquet'
ENERGY_CONS_FILE = f'{OUTPUT_DIR}/energy_consumption.parquet'
BATTERY_TRACE_FILE = f'{OUTPUT_DIR}/battery_trace.parquet'
CENT_CHARGING_EVENTS_FILE = f'{OUTPUT_DIR}/centralized_charging_events.parquet'
LOSS_EVENTS_FILE = f'{OUTPUT_DIR}/loss_events.parquet'

#taxiid
TAXIIDS_FILE = f'{OUTPUT_DIR}/newid_filter.pkl'

MAX_WORKERS = 8   # parallel processing


## Station data


In [41]:
stations_df = pd.read_csv(STATION_CSV, dtype={'station_id': np.int32})
piles_df = pd.read_csv(PILE_CSV, dtype={'station_id': np.int32, 'pile_id': np.int32})
# filter valid piles and stations
max_power = 60
valid_piles_df = piles_df[(piles_df['power'] > 0) & (piles_df['power'] <= max_power)].copy()

stat_pile_df = valid_piles_df.groupby('station_id').agg(
    num_piles=('power', 'count'),
    avg_power=('power', 'mean'),
    max_power=('power', 'max'),
    min_power=('power', 'min')
).reset_index()

# aggregation
stations_df = stations_df.merge(stat_pile_df, on='station_id', how='inner')
stations_df = stations_df[stations_df['num_piles'] > 0].copy()

print(f"there are {len(stations_df)} valid stations")


there are 1478 valid stations


In [42]:
# KDTree of stations
station_coords = stations_df[['longitude', 'latitude']].to_numpy()
station_tree = cKDTree(station_coords)
station_id_lookup = stations_df['station_id'].to_numpy()


# All taxis to process

In [6]:
def get_all_taxiids(tracks_dir):
    """从目录结构中提取所有taxiid"""
    tracks_path = Path(tracks_dir)
    if not tracks_path.exists():
        logging.error(f"轨迹数据目录不存在: {tracks_dir}")
        return []

    taxiids = []
    for item in tracks_path.iterdir():
        if item.is_dir() and item.name.startswith('taxiid='):
            taxiid = item.name.replace('taxiid=', '')
            taxiids.append(taxiid)

    return sorted(taxiids)


In [7]:
all_taxiids = get_all_taxiids(TRACKS_DIR)
logging.info(f"找到 {len(all_taxiids)} 辆车")

2025-11-26 15:15:34,597 | INFO | 找到 19495 辆车


In [6]:
with open(TAXIIDS_FILE, 'rb') as f:
    all_taxiids = pickle.load(f)
logging.info(f"find {len(all_taxiids)} taxis")

2025-11-26 13:43:39,376 | INFO | find 15302 taxis


In [43]:
pickle_path2 = f'output_v5/validid_filter.pkl'
with open(pickle_path2 , 'rb') as f:
    all_taxiids = pickle.load(f)
logging.info(f"find {len(all_taxiids)} taxis")

2025-11-26 17:03:11,988 | INFO | find 15853 taxis


# station_sz filter

In [46]:
sz_stations_df = pd.read_csv('matched_stations_info.csv')
len(sz_stations_df)

1791

In [45]:

def haversine_np(lon1, lat1, lon2, lat2):
    R = 6371000
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Load tables
chargestation_sz = pd.read_csv('chargestation_sz.csv')

# Get coordinate columns
stations_lon = next(col for col in stations_df.columns if any(k in col.lower() for k in ['经度', 'longitude', 'lon']))
stations_lat = next(col for col in stations_df.columns if any(k in col.lower() for k in ['纬度', 'latitude', 'lat']))
sz_lon = next(col for col in chargestation_sz.columns if any(k in col.lower() for k in ['经度', 'longitude', 'lon']))
sz_lat = next(col for col in chargestation_sz.columns if any(k in col.lower() for k in ['纬度', 'latitude', 'lat']))

# Build KDTree for station_information
station_coords = stations_df[[stations_lon, stations_lat]].values
station_tree = cKDTree(station_coords)

# Query nearest station for each chargestation_sz
sz_coords = chargestation_sz[[sz_lon, sz_lat]].values
distances_deg, nearest_indices = station_tree.query(sz_coords, k=1)

# Calculate distances in meters
nearest_station_coords = station_coords[nearest_indices]
distances_m = haversine_np(
    sz_coords[:, 0], sz_coords[:, 1],
    nearest_station_coords[:, 0], nearest_station_coords[:, 1]
)

# Filter: keep only stations with distance >= 200m
THRESHOLD = 200
sz_stations_df = chargestation_sz[distances_m >= THRESHOLD].copy()

print(f"Original: {len(chargestation_sz)} stations")
print(f"Filtered: {len(sz_stations_df)} stations (distance >= {THRESHOLD}m)")


Original: 4423 stations
Filtered: 2746 stations (distance >= 200m)


In [12]:
if 'station_id' not in sz_stations_df.columns:
    sz_stations_df['station_id'] = np.arange(1500,1500+len(sz_stations_df))
sz_stations_df.head()

,name,class,subcalss,lon,lat,province,city,district,station_id
0,杨梅坑专用停车场充电站,汽车相关,充电站,114.571256,22.544423,广东省,深圳市,龙岗区,1500
1,开迈斯汽车公共充电站,汽车相关,充电站,114.568624,22.545040,广东省,深圳市,龙岗区,1501
2,明天新能源公共充电站,汽车相关,充电站,114.518554,22.534409,广东省,深圳市,龙岗区,1502
3,南方和顺汽车公共充电站,汽车相关,充电站,114.528781,22.529354,广东省,深圳市,龙岗区,1503
4,云快充汽车公共充电站,汽车相关,充电站,114.518477,22.534385,广东省,深圳市,龙岗区,1504


# Potential charging at centralized station

# GPS loss events

In [9]:
def detect_GPS_loss(taxiid, tracks_dir, gap_threshold=1800):
    try:
        # read trajectory data
        track_path = os.path.join(tracks_dir, f'taxiid={taxiid}')
        if not os.path.exists(track_path):
            return pd.DataFrame()

        track_files = list(Path(track_path).glob('*.parquet'))
        if not track_files:
            return pd.DataFrame()

        track = pd.concat([pd.read_parquet(f) for f in track_files], ignore_index=True)
        track = track.sort_values('time').reset_index(drop=True)

        if track.empty or len(track) < 2:
            return pd.DataFrame()

        # gap duration
        track['prev_time'] = track['time'].shift()
        track['prev_lon'] = track['lon'].shift()
        track['prev_lat'] = track['lat'].shift()
        track['dt'] = (track['time'] - track['prev_time']).dt.total_seconds()

        #gap duration  >= gap_threshold
        long_gaps = track[track['dt'] >= gap_threshold].copy()

        if long_gaps.empty:
            return pd.DataFrame()

        loss_events = []
        for idx, row in long_gaps.iterrows():
            loss_events.append({
                'taxiid': taxiid,
                'start_time': row['prev_time'],
                'end_time': row['time'],
                'duration_s': row['dt'],
                'start_lon': row['prev_lon'],
                'start_lat': row['prev_lat']
            })

        return pd.DataFrame(loss_events)

    except Exception as e:
        logging.error(f"erroe;{e} for {taxiid}")
        return pd.DataFrame()


def batch_detect_loss_events(all_taxiids, tracks_dir, output_file, gap_threshold=1800, max_workers=8):
    """

    """
    if os.path.exists(output_file):
        logging.info(f"{output_file} already exists")
        return pd.read_parquet(output_file)

    from concurrent.futures import ThreadPoolExecutor, as_completed

    all_loss_events = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(detect_GPS_loss, tid, tracks_dir, gap_threshold): tid
            for tid in all_taxiids
        }

        for future in tqdm(as_completed(futures), total=len(futures), desc="检测专用站充电事件"):
            taxiid = futures[future]
            try:
                result = future.result()
                if not result.empty:
                    all_loss_events.append(result)
            except Exception as e:
                logging.error(f"车辆 {taxiid} 处理失败: {e}")

    if all_loss_events:
        loss_events_df = pd.concat(all_loss_events, ignore_index=True)
        loss_events_df = loss_events_df.sort_values(['taxiid', 'start_time']).reset_index(drop=True)
        loss_events_df.to_parquet(output_file, index=False)
        logging.info(f"find {len(loss_events_df)} GPS loss enents, \
        covering {loss_events_df['taxiid'].nunique()} taxis,saved to {output_file}")
    else:
        logging.warning("No GPS loss events found")
        loss_events_df = pd.DataFrame()

    return loss_events_df

In [48]:
# get GPS loss events
if os.path.exists(LOSS_EVENTS_FILE):
    loss_events_df = pd.read_parquet(LOSS_EVENTS_FILE)
    loss_events_df = loss_events_df[loss_events_df['taxiid'].isin(all_taxiids)].copy()
    logging.info(f"find {len(loss_events_df)} GPS loss events,covering {loss_events_df['taxiid'].nunique()} taxis")

else:
    GAP_THRESHOLD = 1800
    loss_events_df = batch_detect_loss_events(
        all_taxiids,
        TRACKS_DIR,
        LOSS_EVENTS_FILE,
        gap_threshold=GAP_THRESHOLD
    )
    loss_events_df = loss_events_df[loss_events_df['taxiid'].isin(all_taxiids)].copy()
    if not loss_events_df.empty:
        logging.info(f"find {len(loss_events_df)} GPS loss events, \
        covering {loss_events_df['taxiid'].nunique()} taxis")


2025-11-26 17:05:10,117 | INFO | find 315906 GPS loss events,covering 15853 taxis


# Identify charging events at centralized stations

# pre-process

In [49]:
# Define Shenzhen bounding box
SZ_BBOX = {
    'lon_min': 113.7,
    'lon_max': 114.7,
    'lat_min': 22.45,
    'lat_max': 22.85
}

# Filter vehicles - keep only those with all events within Shenzhen
loss_events_df['in_sz'] = (
    loss_events_df['start_lon'].between(SZ_BBOX['lon_min'], SZ_BBOX['lon_max']) &
    loss_events_df['start_lat'].between(SZ_BBOX['lat_min'], SZ_BBOX['lat_max']))

vehicles_with_outside_events = loss_events_df[~loss_events_df['in_sz']]['taxiid'].unique().tolist()
filtered_loss_events  = loss_events_df[~loss_events_df['taxiid'].isin(vehicles_with_outside_events)]
valid_vehicles = filtered_loss_events['taxiid'].unique().tolist()
print(f"Filtered vehicles: {len(valid_vehicles)}")
print(f"Filtered loss events: {len(filtered_loss_events)}")

# process stations
sz_station_coords = sz_stations_df[['lon','lat']].values
sz_station_tree = cKDTree(sz_station_coords)
sz_station_ids =  sz_stations_df['station_id'].values


Filtered vehicles: 15853
Filtered loss events: 315906


In [14]:
all_taxiids = valid_vehicles.copy()

# features of GPS loss events in Shenzhen

In [50]:
def haversine_np(lon1, lat1, lon2, lat2):
    """Calculate distance in meters for inputs in DEGREES"""
    R = 6371000
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# 1. Load & Prep Data
loss_events_df = filtered_loss_events.copy()
# Ensure input to DBSCAN is [lat, lon]
coords_lat_lon = loss_events_df[['start_lat', 'start_lon']].values

BATCH_SIZE = 50000
EPS_RAD = 500 / 6371000
MIN_SAMPLES = 3

# Shenzhen Boundary
SZ_BBOX = {'lon_min': 113.7, 'lon_max': 114.7, 'lat_min': 22.4, 'lat_max': 22.9}

batch_cluster_centers = []
total_noise = 0

# 2. Batch Clustering
n_batches = (len(coords_lat_lon) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Processing {len(coords_lat_lon)} events in {n_batches} batches...")

for i in range(n_batches):
    start = i * BATCH_SIZE
    end = min((i + 1) * BATCH_SIZE, len(coords_lat_lon))
    batch_coords = coords_lat_lon[start:end] # [lat, lon]

    # DBSCAN (metric='haversine' expects [lat, lon] in radians)
    db = DBSCAN(eps=EPS_RAD, min_samples=MIN_SAMPLES, metric='haversine', algorithm='ball_tree')
    labels = db.fit_predict(np.radians(batch_coords))

    total_noise += (labels == -1).sum()

    # Calculate centers for this batch
    unique_labels = set(labels)
    unique_labels.discard(-1)

    for label in unique_labels:
        mask = (labels == label)
        points = batch_coords[mask] # [lat, lon]

        mean_lat = points[:, 0].mean()
        mean_lon = points[:, 1].mean()

        # Filter: Keep only centers within Shenzhen
        if (SZ_BBOX['lon_min'] <= mean_lon <= SZ_BBOX['lon_max'] and
            SZ_BBOX['lat_min'] <= mean_lat <= SZ_BBOX['lat_max']):

            # Store as [lon, lat] for consistency
            batch_cluster_centers.append([mean_lon, mean_lat])

print(f"Found {len(batch_cluster_centers)} valid cluster centers in Shenzhen.")



Processing 315906 events in 7 batches...
Found 1484 valid cluster centers in Shenzhen.


In [51]:
# 3. Merge Nearby Centers (Global)
if len(batch_cluster_centers) > 0:
    batch_centers_arr = np.array(batch_cluster_centers) # [lon, lat]

    # Convert to [lat, lon] radians for merging
    centers_rad = np.radians(batch_centers_arr[:, [1, 0]])

    # Merge centers within 200m
    MERGE_EPS = 200 / 6371000
    db_merge = DBSCAN(eps=MERGE_EPS, min_samples=1, metric='haversine', algorithm='ball_tree')
    merge_labels = db_merge.fit_predict(centers_rad)

    final_centers = []
    unique_merge_labels = set(merge_labels)

    for label in unique_merge_labels:
        mask = (merge_labels == label)
        points = batch_centers_arr[mask] # [lon, lat]

        mean_lon = points[:, 0].mean()
        mean_lat = points[:, 1].mean()
        final_centers.append([mean_lon, mean_lat])

    cluster_centers = np.array(final_centers)
    print(f"Final merged cluster centers: {len(cluster_centers)}")

else:
    cluster_centers = np.array([])
    print("No valid clusters found.")

Final merged cluster centers: 821


In [52]:
import folium

# Create map
m = folium.Map(location=[22.6, 114.2], zoom_start=11, tiles='OpenStreetMap')

# 1. Draw Bounding Box (Blue Rectangle)
bounds = [[SZ_BBOX['lat_min'], SZ_BBOX['lon_min']],
          [SZ_BBOX['lat_max'], SZ_BBOX['lon_max']]]
folium.Rectangle(
    bounds=bounds,
    color="blue",
    weight=2,
    fill=False,
    popup="Valid Range"
).add_to(m)

# 2. Draw Stations (Green Circles, Hollow)
for _, row in sz_stations_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        color='green',
        fill=False,  # Hollow
        weight=2,
        popup=f"Station {row.get('station_id', 'N/A')}"
    ).add_to(m)

# 3. Draw Cluster Centers (Red Circles, Hollow)
for i, center in enumerate(cluster_centers):
    folium.CircleMarker(
        location=[center[1], center[0]], # [lat, lon]
        radius=4,
        color='red',
        fill=False,  # Hollow
        weight=2,
        popup=f"Cluster {i+1}"
    ).add_to(m)

print(f"Map contains: {len(cluster_centers)} clusters, {len(sz_stations_df)} stations")
m

Map contains: 821 clusters, 1791 stations


In [54]:
# 4. Match with Stations
if len(cluster_centers) > 0:
    # Ensure stations are loaded (sz_stations_df)
    sz_station_coords = sz_stations_df[['lon', 'lat']].values
    sz_station_tree = cKDTree(sz_station_coords)

    # Query nearest (inputs are [lon, lat])
    distances_deg, nearest_indices = sz_station_tree.query(cluster_centers, k=1)
    nearest_station_coords = sz_station_coords[nearest_indices]

    # Calculate distances (m)
    distances_m = haversine_np(
        cluster_centers[:, 0], cluster_centers[:, 1],
        nearest_station_coords[:, 0], nearest_station_coords[:, 1]
    )

    print(f"\nDistance Stats (m):")
    print(f"  Mean: {distances_m.mean():.1f}")
    print(f"  Median: {np.median(distances_m):.1f}")
    print(f"  Min: {distances_m.min():.1f}")
    print(f"  Max: {distances_m.max():.1f}")
    print(f"  <200m: {(distances_m < 200).sum()} ({(distances_m < 200).mean()*100:.1f}%)")
    print(f"  <500m: {(distances_m < 500).sum()} ({(distances_m < 500).mean()*100:.1f}%)")
    print(f"  <1000m: {(distances_m < 1000).sum()} ({(distances_m < 1000).mean()*100:.1f}%)")


Distance Stats (m):
  Mean: 634.4
  Median: 375.8
  Min: 13.1
  Max: 14349.9
  <200m: 143 (17.4%)
  <500m: 564 (68.7%)
  <1000m: 749 (91.2%)


# centralized station charging

In [56]:
if os.path.exists(CENT_CHARGING_EVENTS_FILE):
    cent_charging_events = pd.read_parquet(CENT_CHARGING_EVENTS_FILE)
    cent_charging_events = cent_charging_events[cent_charging_events['taxiid'].isin(all_taxiids)].copy()
    print(f"exist {CENT_CHARGING_EVENTS_FILE}, covering {cent_charging_events['taxiid'].nunique()} taxis, and {cent_charging_events['nearest_station_id'].nunique()} stations")
else:
    event_coords = filtered_loss_events[['start_lon', 'start_lat']].values
    distances_deg, nearest_indices = sz_station_tree.query(event_coords, k=1)

    nearest_station_ids = sz_station_ids[nearest_indices]
    nearest_station_coords = sz_station_coords[nearest_indices]

    # Calculate actual distances in meters
    distances_m = haversine_distance(
        filtered_loss_events['start_lon'].values,
        filtered_loss_events['start_lat'].values,
        nearest_station_coords[:, 0],
        nearest_station_coords[:, 1]
    )

    filtered_loss_events['nearest_station_id'] = nearest_station_ids
    filtered_loss_events['distance_to_station_m'] = distances_m.get()

    # Identify centralized charging events (distance < DIST_THRESHOLD(m))
    DIST_THRESHOLD = 500
    filtered_loss_events['is_cent_char'] = filtered_loss_events['distance_to_station_m'] < DIST_THRESHOLD
    cent_charging_events = filtered_loss_events[filtered_loss_events['is_cent_char']].copy()

    print(f"  Events within {DIST_THRESHOLD}m: {len(cent_charging_events)} ({len(cent_charging_events)/len(filtered_loss_events)*100:.2f}%)")

    # Save results
    cent_charging_events.to_parquet(CENT_CHARGING_EVENTS_FILE, index=False)
    print(f"\nSaved centralized charging events to: {CENT_CHARGING_EVENTS_FILE}, covering {cent_charging_events['taxiid'].nunique()} taxis, and {cent_charging_events['nearest_station_id'].nunique()} stations")

exist output_v5/centralized_charging_events.parquet, covering 15805 taxis, and 1742 stations


In [27]:

# Step 1: Get unique station IDs that matched charging events
matched_station_ids = cent_charging_events['nearest_station_id'].unique()

# Step 2: Extract station information for matched stations
# Check if sz_stations_df has 'station_id' column or uses index
if 'station_id' in sz_stations_df.columns:
    matched_stations = sz_stations_df[sz_stations_df['station_id'].isin(matched_station_ids)].copy()
elif sz_stations_df.index.name == 'station_id' or sz_stations_df.index.dtype in [np.int64, np.int32]:
    # If station_id is the index
    matched_stations = sz_stations_df.loc[sz_stations_df.index.isin(matched_station_ids)].copy()
else:
    # If no station_id column, create one from index
    sz_stations_df = sz_stations_df.copy()
    sz_stations_df['station_id'] = sz_stations_df.index
    matched_stations = sz_stations_df[sz_stations_df['station_id'].isin(matched_station_ids)].copy()


print(f"\nExtracted matched stations: {len(matched_stations)}")





Extracted matched stations: 1791


In [28]:
# Step 5: Save result
output_file = 'matched_stations_info.csv'
matched_stations.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\nSaved to: {output_file}")

# Display full info
print(f"\nFull matched stations info:")
print(matched_stations.head())


Saved to: matched_stations_info.csv

Full matched stations info:
              name class subcalss         lon        lat province city  \
1       开迈斯汽车公共充电站  汽车相关      充电站  114.568624  22.545040      广东省  深圳市   
14      比亚迪汽车公共充电站  汽车相关      充电站  114.473199  22.596834      广东省  深圳市   
15      象前充汽车公共充电站  汽车相关      充电站  114.473360  22.598727      广东省  深圳市   
16      深圳供电局公共充电站  汽车相关      充电站  114.478574  22.600642      广东省  深圳市   
32  充电有道长岭村汽车公共充电站  汽车相关      充电站  114.188342  22.552739      广东省  深圳市   

   district  station_id  
1       龙岗区        1501  
14      龙岗区        1514  
15      龙岗区        1515  
16      龙岗区        1516  
32      罗湖区        1520  


In [ ]:
# analysis to be done and visualization: gap loss events temporal-spatio features to determine possible charging, and after matching,
# we keep about 90% loss events that are close to a station (distribution of distance to station).

## Identify charging at distributed public stations

In [29]:
def detect_charging_for_one_taxi(taxiid, tracks_dir, station_tree, station_id_lookup, stations_df):

    try:
        # 读取轨迹数据
        track_path = os.path.join(tracks_dir, f'taxiid={taxiid}')
        if not os.path.exists(track_path):
            return pd.DataFrame()

        # 读取parquet文件（可能有多个分区文件）
        track_files = list(Path(track_path).glob('*.parquet'))
        if not track_files:
            return pd.DataFrame()

        # 读取并合并所有分区
        track = pd.concat([pd.read_parquet(f) for f in track_files], ignore_index=True)

        if track.empty:
            return pd.DataFrame()

        # 按时间排序
        track = track.sort_values('time').reset_index(drop=True)

        # 筛选空载点
        track = track[track['passenger'] == PASSENGER_THR].copy()

        if track.empty:
            return pd.DataFrame()

        # 查询最近充电站（CPU）
        track_coords = track[['lon', 'lat']].to_numpy()
        distances_deg, indices = station_tree.query(track_coords, k=1)
        distances_m = distances_deg * 111320  # 转换为米

        # 筛选在充电站半径内的点
        mask_in_station = distances_m <= R_STATION
        track_in_station = track[mask_in_station].copy()

        if track_in_station.empty:
            return pd.DataFrame()

        # 添加站点ID和距离信息
        track_in_station['nearest_station_id'] = station_id_lookup[indices[mask_in_station]]
        track_in_station['distance_to_station'] = distances_m[mask_in_station]

        # 按站点ID和时间排序
        track_in_station = track_in_station.sort_values(['nearest_station_id', 'time']).reset_index(drop=True)

        # 分段：同一站点内连续的点为一组
        # 如果时间间隔 > T_GAP 或站点ID变化，则断开
        track_in_station['prev_time'] = track_in_station.groupby('nearest_station_id')['time'].shift()
        track_in_station['prev_station'] = track_in_station['nearest_station_id'].shift()

        track_in_station['dt'] = (track_in_station['time'] - track_in_station['prev_time']).dt.total_seconds()
        track_in_station['station_changed'] = (track_in_station['nearest_station_id'] != track_in_station['prev_station'])

        # 标记段的开始（第一个点，或时间间隔过大，或站点变化）
        track_in_station['segment_start'] = (
            track_in_station['prev_time'].isna() |
            (track_in_station['dt'] > T_GAP) |
            track_in_station['station_changed']
        )

        # 分配段ID
        track_in_station['segment_id'] = track_in_station['segment_start'].cumsum()

        # 聚合每个段
        segments = track_in_station.groupby(['nearest_station_id', 'segment_id']).agg(
            start_time=('time', 'min'),
            end_time=('time', 'max'),
            stay_lon=('lon', 'mean'),
            stay_lat=('lat', 'mean'),
            num_points=('time', 'count'),
            avg_distance=('distance_to_station', 'mean')
        ).reset_index()

        # 计算持续时间
        segments['duration_s'] = (segments['end_time'] - segments['start_time']).dt.total_seconds()

        # 筛选满足最小停留时间的段
        charging_events = segments[segments['duration_s'] >= T_STAY].copy()

        # 添加taxiid
        charging_events['taxiid'] = taxiid

        # 选择输出列
        result = charging_events[['taxiid', 'nearest_station_id', 'start_time', 'end_time',
                                   'duration_s', 'stay_lon', 'stay_lat', 'num_points']].copy()

        return result

    except Exception as e:
        logging.error(f"处理车辆 {taxiid} 时出错: {e}")
        return pd.DataFrame()


In [57]:
# 如果文件已存在，可以选择跳过或重新计算
if os.path.exists(DIST_CHARGING_EVENTS_FILE):
    dist_charging_events = pd.read_parquet(DIST_CHARGING_EVENTS_FILE)
    dist_charging_events = dist_charging_events[dist_charging_events['taxiid'].isin(all_taxiids)].copy()
    logging.info(f"充电事件文件已存在: {DIST_CHARGING_EVENTS_FILE},涉及 {dist_charging_events['taxiid'].nunique()} 辆车")

else:
    # 批量处理
    dist_charging_events = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(detect_charging_for_one_taxi, tid, TRACKS_DIR,
                          station_tree, station_id_lookup, stations_df): tid
            for tid in all_taxiids
        }

        for future in tqdm(as_completed(futures), total=len(futures), desc="检测充电事件"):
            taxiid = futures[future]
            try:
                result = future.result()
                if not result.empty:
                    dist_charging_events.append(result)
            except Exception as e:
                logging.error(f"车辆 {taxiid} 处理失败: {e}")

    # 合并所有结果
    if dist_charging_events:
        dist_charging_events = pd.concat(dist_charging_events, ignore_index=True)
        dist_charging_events = dist_charging_events.sort_values(['taxiid', 'start_time']).reset_index(drop=True)
        dist_charging_events = dist_charging_events[dist_charging_events['taxiid'].isin(all_taxiids)].copy()
        # 保存结果
        dist_charging_events.to_parquet(DIST_CHARGING_EVENTS_FILE, index=False)
        logging.info(f"检测到 {len(dist_charging_events)} 个充电事件，涉及 {dist_charging_events['taxiid'].nunique()} 辆车,已保存到 {DIST_CHARGING_EVENTS_FILE}")
    else:
        logging.warning("未检测到任何充电事件")

2025-11-26 17:13:21,382 | INFO | 充电事件文件已存在: output_v5/distributed_charging_events.parquet,涉及 15707 辆车


In [58]:
# ========== 合并公共充电事件和专用站充电事件，用于后续能耗计算 ==========

if not dist_charging_events.empty:
    dist_charges_formatted = pd.DataFrame({
        'taxiid': dist_charging_events['taxiid'],
        'nearest_station_id': dist_charging_events['nearest_station_id'],
        'start_time': dist_charging_events['start_time'],
        'end_time': dist_charging_events['end_time'],
        'duration_s': dist_charging_events['duration_s'],
        'stay_lon': dist_charging_events['stay_lon'],
        'stay_lat': dist_charging_events['stay_lat'],
        'charge_type': 'distributed'
    })


if not cent_charging_events.empty:
    cent_charges_formatted = pd.DataFrame({
        'taxiid': cent_charging_events['taxiid'],
        'nearest_station_id': -1,  # 专用站充电标记为-1
        'start_time': cent_charging_events['start_time'],
        'end_time': cent_charging_events['end_time'],
        'duration_s': cent_charging_events['duration_s'],
        'stay_lon': cent_charging_events['start_lon'],
        'stay_lat': cent_charging_events['start_lat'],
        'charge_type': 'centralized'
    })


if not dist_charges_formatted.empty and not cent_charges_formatted.empty:
    all_charging_events = pd.concat([
        dist_charges_formatted,
        cent_charges_formatted
    ], ignore_index=True)
    logging.info(f"合并成功，总事件数: {len(all_charging_events)}")
else:
    logging.warning("没有找到任何充电事件")

if not all_charging_events.empty:
    all_charging_events = all_charging_events.sort_values(['taxiid', 'start_time']).reset_index(drop=True)
    print(f"总事件数: {len(all_charging_events)}")
    print(f"\n涉及车辆数: {all_charging_events['taxiid'].nunique()}")


2025-11-26 17:13:28,230 | INFO | 合并成功，总事件数: 417051


总事件数: 417051

涉及车辆数: 15853


In [ ]:
# waiting time, charging time distribution, frequence

## Queue model


In [32]:
def simulate_queue(df):

    if df.empty:
        return pd.DataFrame()

    k = int(df['num_piles'].iloc[0])  # 充电桩数量
    busy_chargers_heap = []  # 最小堆，存储正在充电的车辆结束时间
    results = []

    for _, row in df.iterrows():
        arrival_time = row['start_time']
        departure_time = row['end_time']

        # 移除在当前车辆到达前已空闲的充电桩
        while busy_chargers_heap and busy_chargers_heap[0] <= arrival_time:
            heappop(busy_chargers_heap)

        # 检查是否有可用充电桩
        if len(busy_chargers_heap) < k:
            # 有空闲充电桩，无需等待
            wait_seconds = 0
            charge_start_time = arrival_time
        else:
            # 所有充电桩都忙，需要等待
            earliest_free_time = heappop(busy_chargers_heap)
            wait_seconds = (earliest_free_time - arrival_time).total_seconds()
            charge_start_time = earliest_free_time

        # 判断是否放弃充电
        stay_duration = (departure_time - arrival_time).total_seconds()
        gave_up = (wait_seconds >= stay_duration)

        # 计算实际充电时长
        if gave_up:
            actual_charge_seconds = 0
            # 如果放弃，把之前弹出的时间放回去
            if 'earliest_free_time' in locals() and wait_seconds > 0:
                heappush(busy_chargers_heap, earliest_free_time)
        else:
            actual_charge_seconds = (departure_time - charge_start_time).total_seconds()
            # 该车辆占用充电桩直到离开时间
            heappush(busy_chargers_heap, departure_time)

        results.append({
            'taxiid': row['taxiid'],
            'nearest_station_id': row['nearest_station_id'],
            'arrive_time': arrival_time,
            'leave_time': departure_time,
            'wait_dur': wait_seconds,
            'giveup': gave_up,
            'charge_dur': actual_charge_seconds
        })

    return pd.DataFrame(results)


In [59]:
# 执行排队模拟
if os.path.exists(CHAR_QUEUE_FILE):
    char_queue_df = pd.read_parquet(CHAR_QUEUE_FILE)
    char_queue_df = char_queue_df[char_queue_df['taxiid'].isin(all_taxiids)].copy()
    logging.info(f"已加载排队模拟结果: {len(char_queue_df)} 条记录")

elif not dist_charging_events.empty:
    # 合并站点信息（获取num_piles）
    dist_charging_events = dist_charging_events[dist_charging_events['taxiid'].isin(all_taxiids)].copy()
    queue_input = dist_charging_events.merge(
        stations_df[['station_id', 'num_piles']],
        left_on='nearest_station_id',
        right_on='station_id',
        how='left'
    )

    # 按站点和时间排序
    queue_input = queue_input.sort_values(['nearest_station_id', 'start_time'], kind='mergesort')

    # 对每个站点进行排队模拟
    all_queue_results = []
    for station_id, group in tqdm(queue_input.groupby('nearest_station_id'), desc="排队模拟"):
        result = simulate_queue(group)
        if not result.empty:
            all_queue_results.append(result)

    # 合并结果
    if all_queue_results:
        char_queue_df = pd.concat(all_queue_results, ignore_index=True)
        char_queue_df = char_queue_df.sort_values(['taxiid', 'arrive_time']).reset_index(drop=True)
        # 保存结果
        char_queue_df.to_parquet(CHAR_QUEUE_FILE, index=False)
        logging.info(f"排队模拟完成，结果已保存到 {CHAR_QUEUE_FILE}")

        # 基本统计
        avg_wait = char_queue_df['wait_dur'].mean()
        giveup_rate = char_queue_df['giveup'].mean()
        logging.info(f"平均等待时间: {avg_wait:.2f} 秒 ({avg_wait/60:.2f} 分钟)")
        logging.info(f"放弃率: {giveup_rate:.2%}")
    else:
        logging.warning("排队模拟未产生结果")
        char_queue_df = pd.DataFrame()
else:
    logging.warning("充电事件数据不存在，无法进行排队模拟")


排队模拟: 100%|██████████| 1104/1104 [00:04<00:00, 240.27it/s]
2025-11-26 17:14:21,877 | INFO | 排队模拟完成，结果已保存到 output_v5/char_queue.parquet
2025-11-26 17:14:21,879 | INFO | 平均等待时间: 325.46 秒 (5.42 分钟)
2025-11-26 17:14:21,880 | INFO | 放弃率: 7.84%


## Energy consumption and SOC trajectory

### energy consumption


In [34]:

def calculate_energy_between_charges_v2(tracks_dir, charging_events_taxi):

    try:
        # 从充电事件中获取taxiid
        if charging_events_taxi.empty or len(charging_events_taxi) < 2:
            return pd.DataFrame()

        taxiid = charging_events_taxi.iloc[0]['taxiid']

        # 读取轨迹数据
        track_path = os.path.join(tracks_dir, f'taxiid={taxiid}')
        if not os.path.exists(track_path):
            return pd.DataFrame()

        track_files = list(Path(track_path).glob('*.parquet'))
        if not track_files:
            return pd.DataFrame()

        # 读取轨迹（使用cudf加速）
        track = cudf.concat([cudf.read_parquet(f) for f in track_files], ignore_index=True)

        # 过滤zoneid < 0的点
        track = track[track['zoneid'] >= 0].copy()
        if track.empty:
            return pd.DataFrame()

        track = track.sort_values('time').reset_index(drop=True)

        # 计算相邻点之间的距离和时间差（向量化操作）
        track['prev_lon'] = track['lon'].shift()
        track['prev_lat'] = track['lat'].shift()
        track['prev_time'] = track['time'].shift()

        # 填充第一行的NaN（用自身值填充）
        track['prev_lon'] = track['prev_lon'].fillna(track['lon'])
        track['prev_lat'] = track['prev_lat'].fillna(track['lat'])
        track['prev_time'] = track['prev_time'].fillna(track['time'])

        # 计算时间差和距离
        track['dt'] = (track['time'] - track['prev_time']).dt.total_seconds()
        track['dist'] = haversine_distance(
            track['lon'], track['lat'],
            track['prev_lon'], track['prev_lat']
        )

        # 过滤异常跳点（速度 > 20 m/s，约72 km/h，对于城市出租车来说异常）
        track = track[track['dist'] / track['dt'].clip(lower=1) <= 15].copy()

        # 标记怠速点（速度 <= 5 km/h）
        track['idle'] = (track['velocity'] <= 5).astype('int8')

        # 转换为pandas（只在需要时转换）
        track_pd = track[['time', 'dt', 'dist', 'idle']].to_pandas()

        # 释放GPU内存
        del track
        cp._default_memory_pool.free_all_blocks()

        # 对每对相邻充电事件计算能耗
        results = []
        charging_events_sorted = charging_events_taxi.sort_values('start_time').reset_index(drop=True)

        for i in range(1, len(charging_events_sorted)):
            prev_end = charging_events_sorted.iloc[i - 1]['end_time']
            curr_start = charging_events_sorted.iloc[i]['start_time']
            curr_charge_type = charging_events_sorted.iloc[i]['charge_type']

            # 提取两次充电之间的轨迹段（使用布尔索引，更高效）
            mask = (track_pd['time'] > prev_end) & (track_pd['time'] <= curr_start)
            segment = track_pd[mask].copy()

            if segment.empty:
                continue

            # 检查数据缺失（时间间隔过大）
            large_gaps_mask = segment['dt'] > MAX_GAP_SECONDS
            missing_s = segment.loc[large_gaps_mask, 'dt'].sum() if large_gaps_mask.any() else 0

            # 只计算有数据的部分
            valid_segment = segment[~large_gaps_mask]
            if valid_segment.empty:
                continue

            # 计算行驶和怠速时间（向量化操作）
            drive_s = int(valid_segment.loc[valid_segment['idle'] == 0, 'dt'].sum())
            idle_s = int(valid_segment.loc[valid_segment['idle'] == 1, 'dt'].sum())
            gap_s = drive_s + idle_s

            # 计算距离和能耗
            distance_km = valid_segment['dist'].sum() / 1000
            energy_used_kWh = distance_km * CONS_KWH_PER_KM

            results.append({
                'taxiid': taxiid,
                'charge_start': curr_start,
                'prev_end': prev_end,
                'gap_s': gap_s,
                'drive_s': drive_s,
                'idle_s': idle_s,
                'distance_km': distance_km,
                'energy_used_kWh': energy_used_kWh,
                'missing_s': missing_s,
                'charge_type': curr_charge_type
            })

        return pd.DataFrame(results)

    except Exception as e:
        logging.error(f"计算车辆 {taxiid if 'taxiid' in locals() else 'unknown'} 能耗时出错: {e}")
        import traceback
        logging.error(traceback.format_exc())
        return pd.DataFrame()



In [60]:

# ========== 批量计算所有车辆的能耗（使用合并后的完整充电事件） ==========

# 如果文件已存在，可以选择跳过或重新计算
if os.path.exists(ENERGY_CONS_FILE):
    energy_cons_df = pd.read_parquet(ENERGY_CONS_FILE)
    energy_cons_df = energy_cons_df[energy_cons_df['taxiid'].isin(all_taxiids)].copy()
    logging.info(f"能耗数据文件已存在: {ENERGY_CONS_FILE}")

else:
    all_energy_results = []
    all_charging_events =all_charging_events[all_charging_events['taxiid'].isin(all_taxiids)].copy()
    # 按车辆分组处理（使用groupby，更高效）
    grouped = all_charging_events.groupby('taxiid')

    for taxiid, ce_group in tqdm(grouped, total=grouped.ngroups, desc="计算能耗（含专用站充电）"):
        if len(ce_group) < 2:  # 至少需要2次充电才能计算间隔
            continue

        result = calculate_energy_between_charges_v2(TRACKS_DIR, ce_group)
        if not result.empty:
            all_energy_results.append(result)

        # 每处理100辆车释放一次GPU内存（减少释放频率，提高效率）
        if len(all_energy_results) % 100 == 0:
            cp._default_memory_pool.free_all_blocks()

    # 最终释放GPU内存
    cp._default_memory_pool.free_all_blocks()

    if all_energy_results:
        energy_cons_df = pd.concat(all_energy_results, ignore_index=True)
        energy_cons_df = energy_cons_df.sort_values(['taxiid', 'charge_start']).reset_index(drop=True)

        # 保存结果
        energy_cons_df.to_parquet(ENERGY_CONS_FILE, index=False)
        logging.info(f"能耗计算完成，结果已保存到 {ENERGY_CONS_FILE}")
    else:
        logging.warning("未计算出任何能耗数据")



2025-11-26 17:14:35,601 | INFO | 能耗数据文件已存在: output_v5/energy_consumption.parquet


# 6.2 SOC 轨迹

In [61]:

battery_trace = all_charging_events.copy()

# 2. 分离公共和专用站充电事件
public_mask = battery_trace['charge_type'] == 'distributed'
dedicated_mask = battery_trace['charge_type'] == 'centralized'

battery_trace_public = battery_trace[public_mask].copy()
battery_trace_dedicated = battery_trace[dedicated_mask].copy()

# 3. 处理公共充电事件：合并排队结果和站点功率
if not battery_trace_public.empty:
    battery_trace_public = battery_trace_public.merge(
        char_queue_df[['taxiid', 'arrive_time', 'charge_dur', 'wait_dur', 'giveup']],
        left_on=['taxiid', 'start_time'],
        right_on=['taxiid', 'arrive_time'],
        how='left'
    )

    # 合并站点平均功率
    battery_trace_public = battery_trace_public.merge(
        stations_df[['station_id', 'avg_power']],
        left_on='nearest_station_id',
        right_on='station_id',
        how='left'
    )
    battery_trace_public = battery_trace_public.drop(columns=['station_id'], errors='ignore')

# 4. 处理专用站充电事件：设置默认值
if not battery_trace_dedicated.empty:
    battery_trace_dedicated['arrive_time'] = battery_trace_dedicated['start_time']
    battery_trace_dedicated['charge_dur'] = battery_trace_dedicated['duration_s']  # 全部都是充电时间
    battery_trace_dedicated['wait_dur'] = 0  # 专用站充电没有等待
    battery_trace_dedicated['giveup'] = False  # 专用站充电不放弃
    battery_trace_dedicated['avg_power'] = 43.0  # 专用站充电功率统一为43kW

# 5. 合并公共和专用站充电事件
if not battery_trace_public.empty and not battery_trace_dedicated.empty:
    # 统一列名
    common_cols = ['taxiid', 'nearest_station_id', 'start_time', 'end_time',
                  'duration_s', 'stay_lon', 'stay_lat', 'charge_type',
                  'arrive_time', 'charge_dur', 'wait_dur', 'giveup', 'avg_power']

    battery_trace = pd.concat([
        battery_trace_public[common_cols],
        battery_trace_dedicated[common_cols]
    ], ignore_index=True)


if battery_trace.empty:
    logging.warning("没有充电事件数据，无法计算SOC轨迹")
else:
    # 6. 按车辆和时间排序
    battery_trace = battery_trace.sort_values(['taxiid', 'start_time']).reset_index(drop=True)
    battery_trace['prev_end'] = battery_trace.groupby('taxiid')['end_time'].shift()

    # 7. 合并能耗数据（两次充电之间的能耗，包含charge_type）
    battery_trace = battery_trace.merge(
        energy_cons_df[['taxiid', 'charge_start', 'energy_used_kWh', 'missing_s',
            'gap_s', 'drive_s', 'idle_s', 'distance_km', 'charge_type']],
        left_on=['taxiid', 'start_time'],
        right_on=['taxiid', 'charge_start'],
        how='left',
        suffixes=('', '_from_energy')
    )

    # 使用能耗表中的charge_type（更准确，因为它标记的是当前充电事件的类型）
    battery_trace['charge_type'] = battery_trace['charge_type_from_energy'].fillna(battery_trace['charge_type'])
    battery_trace = battery_trace.drop(columns=['charge_type_from_energy', 'charge_start'], errors='ignore')

    # 8. 初始化能耗数据
    battery_trace['energy_used_kWh'] = battery_trace['energy_used_kWh'].fillna(0)

    # 9. 逐车递推SOC（先计算SOC_before，再计算充入电量）
    def calculate_soc_trace_v2(group):
        soc_before_list = []
        soc_after_list = []
        soc_prev = CAP_KWH  # 初始满电

        for _, row in group.iterrows():
            # 计算充电前SOC
            soc_before = max(0, soc_prev - row['energy_used_kWh'])
            soc_before_list.append(soc_before)

            # 计算充入电量（基于充电前SOC）
            charge_dur_hours = row['charge_dur'] / 3600

            # 公共充电：放弃充电的功率为0，否则使用站平均功率
            # 专用站充电：功率统一为43kW
            if row['giveup'] == True:
                power_kw = 0
            else:
                power_kw = row['avg_power'] if pd.notna(row['avg_power']) else 0

            # 计算理论充入电量
            energy_in_theoretical = charge_dur_hours * power_kw * CHG_EFFICIENCY

            # 限制充入电量：不能超过（满电量 - 充电前电量）
            max_energy_in = CAP_KWH - soc_before
            energy_in = min(energy_in_theoretical, max_energy_in)

            # 计算充电后SOC
            soc_after = min(CAP_KWH, soc_before + energy_in)
            soc_after_list.append(soc_after)

            soc_prev = soc_after

        group['SOC_before_kWh'] = soc_before_list
        group['SOC_after_kWh'] = soc_after_list
        group['energy_in_kWh'] = [soc_after_list[i] - soc_before_list[i] for i in range(len(soc_before_list))]
        return group

    battery_trace = battery_trace.groupby('taxiid', group_keys=False).apply(calculate_soc_trace_v2)

    # 10. 计算百分比
    battery_trace['SOC_before_pct'] = battery_trace['SOC_before_kWh'] / CAP_KWH * 100
    battery_trace['SOC_after_pct'] = battery_trace['SOC_after_kWh'] / CAP_KWH * 100

    # 11. 选择输出列（包含charge_type）
    output_cols = [
        'taxiid', 'charge_type', 'nearest_station_id', 'start_time', 'end_time',
        'SOC_before_kWh', 'SOC_after_kWh', 'SOC_before_pct', 'SOC_after_pct',
        'energy_used_kWh', 'energy_in_kWh',
        'distance_km', 'drive_s', 'idle_s', 'gap_s', 'missing_s',
        'charge_dur', 'wait_dur', 'giveup'
    ]
    output_cols = [col for col in output_cols if col in battery_trace.columns]

    battery_trace_output = battery_trace[output_cols].copy()
    battery_trace_output.to_parquet(BATTERY_TRACE_FILE, index=False)


/tmp/ipykernel_1712/2481420815.py:112: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  battery_trace = battery_trace.groupby('taxiid', group_keys=False).apply(calculate_soc_trace_v2)


In [62]:
# 假设 battery_trace_output 是你包含SOC轨迹的DataFrame

# 1. 找出所有存在 "充电前SOC < 1%" 记录的车辆ID (taxiid)
#    首先，筛选出所有不符合条件的记录
low_soc_events = battery_trace_output[battery_trace_output['SOC_before_pct'] < 0.1]

#    然后，获取这些记录对应的所有唯一的 taxiid
taxiids_to_exclude = low_soc_events['taxiid'].unique()

# 2. 从原始DataFrame中剔除这些车辆的所有记录
#    使用 .isin() 方法来判断每一行的 'taxiid' 是否在我们要排除的列表中
#    前面的 ~ 符号表示 "取反"，即保留那些 taxiid 不在排除列表中的行
filtered_df = battery_trace_output[~battery_trace_output['taxiid'].isin(taxiids_to_exclude)].copy()

# 3. 打印统计信息，验证筛选结果
original_taxi_count = battery_trace_output['taxiid'].nunique()
filtered_taxi_count = filtered_df['taxiid'].nunique()
removed_taxi_count = original_taxi_count - filtered_taxi_count

print("原始DataFrame信息：")
print(f"  总车辆数: {original_taxi_count}")
print(f"  总事件数: {len(battery_trace_output)}")

print(f"\n需要排除的车辆数: {len(taxiids_to_exclude)}")
print(f"  这些车辆是: {taxiids_to_exclude}")

print("\n筛选后DataFrame信息：")
print(f"  剩余车辆数: {filtered_taxi_count}")
print(f"  剩余事件数: {len(filtered_df)}")

print(f"\n共剔除了 {removed_taxi_count} 辆车的所有数据。")

# 新的DataFrame 'filtered_df' 现在可以用于后续分析
# 可以查看一下筛选后的结果
print("\n筛选后的DataFrame示例：")
print(filtered_df.head())

原始DataFrame信息：
  总车辆数: 15853
  总事件数: 417051

需要排除的车辆数: 0
  这些车辆是: []

筛选后DataFrame信息：
  剩余车辆数: 15853
  剩余事件数: 417051

共剔除了 0 辆车的所有数据。

筛选后的DataFrame示例：
      taxiid  charge_type  nearest_station_id          start_time  \
0  UUUB0C0M7  centralized                  -1 2020-01-01 04:17:29   
1  UUUB0C0M7  centralized                  -1 2020-01-01 16:18:31   
2  UUUB0C0M7  distributed                2266 2020-01-02 01:21:09   
3  UUUB0C0M7  centralized                  -1 2020-01-02 04:16:24   
4  UUUB0C0M7  centralized                  -1 2020-01-02 16:09:34   

             end_time  SOC_before_kWh  SOC_after_kWh  SOC_before_pct  \
0 2020-01-01 05:48:13       80.000000       80.00000      100.000000   
1 2020-01-01 17:47:47       48.762003       80.00000       60.952504   
2 2020-01-02 01:35:24       57.535850       59.19835       71.919813   
3 2020-01-02 05:41:39       52.392484       80.00000       65.490605   
4 2020-01-02 17:32:36       50.046223       80.00000       62.557779   



In [38]:
filterids = filtered_df['taxiid'].unique().tolist()

In [39]:
pickle_path2 = f'output_v5/validid_filter.pkl'
with open(pickle_path2, 'wb') as f:  # 注意是 'wb' (write binary)
    # 使用 pickle.dump() 将列表写入文件
    pickle.dump(filterids, f)

print(f"列表已成功保存为 Pickle 文件: {pickle_path2}")

列表已成功保存为 Pickle 文件: output_v5/validid_filter.pkl


In [50]:
target = cent_charging_events[cent_charging_events['taxiid'].isin(filterids)].copy()
use_sz_stats = target['nearest_station_id'].unique().tolist()

In [51]:
len(use_sz_stats)

2809

In [ ]:
pickle_path2 = f'output_v4/id_filter.pkl'
with open(pickle_path2, 'rb') as f:  # 注意是 'wb' (write binary)
    # 使用 pickle.dump() 将列表写入文件
    filterids= pickle.load(f)

## 7. 统计分析和可视化

### 7.1 基本统计指标


In [ ]:
# ========== 1. Load filtered vehicle IDs ==========
all_taxiids = filterids.copy()

battery_trace_output = pd.read_parquet('output_v4/battery_trace_p.parquet')

print(f"Filtered vehicle count: {len(all_taxiids)}")

# ========== 2. Load charging data ==========
CHAR_QUEUE_FILE = 'output_v4/char_queue_p.parquet'
PRIVATE_GAP_FILE = 'output_v4/private_gap_charging_events_p.parquet'
TRACKS_DIR = '../data/taxi_trajectories.parquet'

# Load public charging queue data
char_queue_df = pd.read_parquet(CHAR_QUEUE_FILE)
char_queue_df = char_queue_df[char_queue_df['taxiid'].isin(all_taxiids)].copy()

# Load dedicated charging data
dedicated_charges_df = pd.read_parquet(PRIVATE_GAP_FILE)
dedicated_charges_df = dedicated_charges_df[dedicated_charges_df['taxiid'].isin(all_taxiids)].copy()

print(f"Public charging events: {len(char_queue_df)}")
print(f"Dedicated charging events: {len(dedicated_charges_df)}")


In [ ]:
# ========== Visualization 1: Charging Time and Waiting Time Distribution at Public Stations ==========

print("\n" + "="*60)
print("Visualization 1: Charging Time and Waiting Time Distribution")
print("="*60)

# Convert to minutes
char_queue_df['charging_time_min'] = char_queue_df['charge_dur'] / 60
char_queue_df['waiting_time_min'] = char_queue_df['wait_dur'] / 60

# ========== Configurable thresholds ==========
CHARGING_MAX_THRESHOLD = 80  # Change this to 60, 70, 80, etc. for charging time
WAITING_MAX_THRESHOLD = 60   # Change this to 50, 60, 70, etc. for waiting time

CHARGING_BIN_SIZE = 10  # 10-minute bins
WAITING_BIN_SIZE = 10    # 5-minute bins

# Determine bins for charging time - last bin is ">threshold"
charging_bins = list(np.arange(0, CHARGING_MAX_THRESHOLD, CHARGING_BIN_SIZE)) + [np.inf]
# Example: if threshold=70, bins = [0, 10, 20, 30, 40, 50, 60, 70, inf]

# Determine bins for waiting time - last bin is ">threshold"
waiting_bins = list(np.arange(0, WAITING_MAX_THRESHOLD, WAITING_BIN_SIZE)) + [np.inf]
# Example: if threshold=60, bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, inf]

# Charging time distribution
plt.figure(figsize=(8, 5))
charging_counts, charging_edges = np.histogram(char_queue_df['charging_time_min'], bins=charging_bins)
charging_percentages = charging_counts / len(char_queue_df) * 100

# Create labels - last one is ">threshold"
charging_labels = []
for i in range(len(charging_counts)):
    if charging_edges[i+1] == np.inf:
        charging_labels.append(f">{int(charging_edges[i])}")
    else:
        charging_labels.append(f"{int(charging_edges[i])}-{int(charging_edges[i+1])}")

plt.bar(range(len(charging_counts)), charging_percentages, color='steelblue', alpha=0.7, edgecolor='black')
plt.xlabel('Charging Time (minutes)', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.title('Distribution of Charging Time at Public Charging Stations', fontsize=14, fontweight='bold')
plt.xticks(range(len(charging_labels)), charging_labels, rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Add percentage labels on ALL bars (even if small)
for i, (count, pct) in enumerate(zip(charging_counts, charging_percentages)):
    if count > 0:  # Only show if there are events in this bin
        plt.text(i, pct + 0.3, f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)
    else:
        # Show 0% for empty bins
        plt.text(i, 0.5, '0.0%', ha='center', va='bottom', fontsize=9, color='gray', alpha=0.6)

plt.tight_layout()
plt.show()

# Waiting time distribution
plt.figure(figsize=(8, 5))
waiting_counts, waiting_edges = np.histogram(char_queue_df['waiting_time_min'], bins=waiting_bins)
waiting_percentages = waiting_counts / len(char_queue_df) * 100

# Create labels - last one is ">threshold"
waiting_labels = []
for i in range(len(waiting_counts)):
    if waiting_edges[i+1] == np.inf:
        waiting_labels.append(f">{int(waiting_edges[i])}")
    else:
        waiting_labels.append(f"{int(waiting_edges[i])}-{int(waiting_edges[i+1])}")

plt.bar(range(len(waiting_counts)), waiting_percentages, color='coral', alpha=0.7, edgecolor='black')
plt.xlabel('Waiting Time (minutes)', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.title('Distribution of Waiting Time at Public Charging Stations', fontsize=14, fontweight='bold')
plt.xticks(range(len(waiting_labels)), waiting_labels, rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Add percentage labels on ALL bars (even if small)
for i, (count, pct) in enumerate(zip(waiting_counts, waiting_percentages)):
    if count > 0:  # Only show if there are events in this bin
        plt.text(i, pct + 0.3, f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)
    else:
        # Show 0% for empty bins
        plt.text(i, 0.5, '0.0%', ha='center', va='bottom', fontsize=9, color='gray', alpha=0.6)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSummary Statistics:")
print(f"Charging Time:")
print(f"  Total events: {len(char_queue_df)}")
print(f"  Events > {CHARGING_MAX_THRESHOLD} min: {(char_queue_df['charging_time_min'] > CHARGING_MAX_THRESHOLD).sum()} "
      f"({(char_queue_df['charging_time_min'] > CHARGING_MAX_THRESHOLD).sum()/len(char_queue_df)*100:.2f}%)")
print(f"Waiting Time:")
print(f"  Total events: {len(char_queue_df)}")
print(f"  Events > {WAITING_MAX_THRESHOLD} min: {(char_queue_df['waiting_time_min'] > WAITING_MAX_THRESHOLD).sum()} "
      f"({(char_queue_df['waiting_time_min'] > WAITING_MAX_THRESHOLD).sum()/len(char_queue_df)*100:.2f}%)")

In [ ]:
# ========== Visualization 2: Dedicated Station Charging - Time and Spatial Distribution ==========
print("\n" + "="*60)
print("Visualization 2: Dedicated Station Charging - Time and Spatial Clustering")
print("="*60)

# Ensure datetime
dedicated_charges_df['start_time'] = pd.to_datetime(dedicated_charges_df['start_time'])
dedicated_charges_df['hour'] = dedicated_charges_df['start_time'].dt.hour

# Time distribution
plt.figure(figsize=(14, 6))
hour_counts = dedicated_charges_df['hour'].value_counts().sort_index()
hour_percentages = hour_counts / len(dedicated_charges_df) * 100

plt.bar(hour_counts.index, hour_percentages.values, color='darkgreen', alpha=0.7, edgecolor='black')
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.title('Temporal Distribution of Dedicated Station Charging Events', fontsize=14, fontweight='bold')
plt.xticks(range(24))
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# ========== Spatial Analysis: Charging Points per Vehicle (with 500m merging) ==========
from sklearn.cluster import DBSCAN

def haversine_distance(lon1, lat1, lon2, lat2):
    """Calculate distance in meters"""
    R = 6371000
    lon1_rad, lat1_rad, lon2_rad, lat2_rad = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2_rad - lon1_rad, lat2_rad - lat1_rad
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Merge points within 500m for each vehicle
MERGE_DISTANCE_M = 500  # 500 meters
MIN_SAMPLES = 1  # At least 1 point to form a cluster

points_per_vehicle_list = []

for taxiid, vehicle_data in dedicated_charges_df.groupby('taxiid'):
    coords = vehicle_data[['start_lon', 'start_lat']].values

    if len(coords) == 0:
        continue

    # Convert to radians for DBSCAN
    coords_rad = np.radians(coords)
    eps_rad = MERGE_DISTANCE_M / 6371000  # Convert 500m to radians

    # DBSCAN clustering: points within 500m are merged into one cluster
    dbscan = DBSCAN(eps=eps_rad, min_samples=MIN_SAMPLES, algorithm='ball_tree', metric='haversine')
    cluster_labels = dbscan.fit_predict(coords_rad)

    # Count unique clusters (excluding noise points, but each noise point counts as 1 point)
    unique_clusters = set(cluster_labels)
    if -1 in unique_clusters:
        unique_clusters.remove(-1)
        # Noise points are individual points
        n_noise = (cluster_labels == -1).sum()
        n_merged_points = len(unique_clusters) + n_noise
    else:
        n_merged_points = len(unique_clusters)

    points_per_vehicle_list.append({
        'taxiid': taxiid,
        'num_points': n_merged_points,
        'num_events': len(vehicle_data)
    })

points_per_vehicle = pd.DataFrame(points_per_vehicle_list)
points_per_vehicle = points_per_vehicle.sort_values('num_points', ascending=False)

print(f"\nSpatial Clustering Analysis (Points merged within {MERGE_DISTANCE_M}m):")
print(f"  Total vehicles: {len(points_per_vehicle)}")
print(f"  Total charging events: {len(dedicated_charges_df)}")
print(f"  Average merged points per vehicle: {points_per_vehicle['num_points'].mean():.2f}")
print(f"  Median merged points per vehicle: {points_per_vehicle['num_points'].median():.2f}")
print(f"  Min merged points per vehicle: {points_per_vehicle['num_points'].min()}")
print(f"  Max merged points per vehicle: {points_per_vehicle['num_points'].max()}")

# Calculate how many vehicles cover 90% of charging events
# Sort by number of events (not points) for coverage calculation
events_per_vehicle = dedicated_charges_df.groupby('taxiid').size().reset_index(name='num_events')
events_per_vehicle = events_per_vehicle.sort_values('num_events', ascending=False)
cumulative_events = events_per_vehicle['num_events'].cumsum()
total_events = events_per_vehicle['num_events'].sum()
coverage_90_threshold = total_events * 0.8

vehicles_needed_90 = (cumulative_events <= coverage_90_threshold).sum() + 1
if vehicles_needed_90 > len(events_per_vehicle):
    vehicles_needed_90 = len(events_per_vehicle)

# Calculate average merged points for top vehicles covering 90%
top_vehicles = events_per_vehicle.head(vehicles_needed_90)['taxiid'].tolist()
top_vehicles_points = points_per_vehicle[points_per_vehicle['taxiid'].isin(top_vehicles)]['num_points'].mean()

print(f"\nCoverage Analysis:")
print(f"  Vehicles needed to cover 90% of events: {vehicles_needed_90} ({vehicles_needed_90/len(events_per_vehicle)*100:.2f}% of vehicles)")
print(f"  Average merged points per vehicle (top {vehicles_needed_90} vehicles): {top_vehicles_points:.2f}")

# Distribution of merged points per vehicle
plt.figure(figsize=(12, 6))

# Histogram of merged points per vehicle
max_points = points_per_vehicle['num_points'].max()
bins = min(50, max_points + 1)  # Use appropriate number of bins
plt.hist(points_per_vehicle['num_points'], bins=bins, color='steelblue', alpha=0.7, edgecolor='black')
plt.xlabel('Number of Merged Charging Points per Vehicle', fontsize=12)
plt.ylabel('Number of Vehicles', fontsize=12)
plt.title(f'Distribution of Dedicated Charging Points per Vehicle\n(Points within {MERGE_DISTANCE_M}m merged)', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# Add vertical lines for mean and median
mean_points = points_per_vehicle['num_points'].mean()
median_points = points_per_vehicle['num_points'].median()
plt.axvline(mean_points, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_points:.1f}')
plt.axvline(median_points, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_points:.1f}')
plt.legend(fontsize=10)

plt.tight_layout()
plt.show()

# Additional statistics
print(f"\nDistribution Statistics:")
print(f"  25th percentile: {points_per_vehicle['num_points'].quantile(0.25):.1f} points")
print(f"  75th percentile: {points_per_vehicle['num_points'].quantile(0.75):.1f} points")
print(f"  90th percentile: {points_per_vehicle['num_points'].quantile(0.90):.1f} points")
print(f"  95th percentile: {points_per_vehicle['num_points'].quantile(0.95):.1f} points")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

# ========== Visualization 2: Dedicated Station Charging - Time and Spatial Distribution ==========
print("\n" + "="*60)
print("Visualization 2: Dedicated Station Charging - Time and Spatial Clustering")
print("="*60)

# Ensure datetime
dedicated_charges_df['start_time'] = pd.to_datetime(dedicated_charges_df['start_time'])
dedicated_charges_df['hour'] = dedicated_charges_df['start_time'].dt.hour

# Time distribution
plt.figure(figsize=(14, 6))
hour_counts = dedicated_charges_df['hour'].value_counts().sort_index()
hour_percentages = hour_counts / len(dedicated_charges_df) * 100

plt.bar(hour_counts.index, hour_percentages.values, color='darkgreen', alpha=0.7, edgecolor='black')
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.title('Temporal Distribution of Dedicated Station Charging Events', fontsize=14, fontweight='bold')
plt.xticks(range(24))
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# ========== Calculate Spatial and Temporal Clustering Scores for Each Vehicle ==========
MERGE_DISTANCE_M = 500  # 500 meters
SPATIAL_THRESHOLD_RATIO = 0.9  # 80% of events in main clusters
TIME_THRESHOLD_RATIO = 0.8     # 80% of events in main time slots

def haversine_distance(lon1, lat1, lon2, lat2):
    """Calculate distance in meters"""
    R = 6371000
    lon1_rad, lat1_rad, lon2_rad, lat2_rad = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2_rad - lon1_rad, lat2_rad - lat1_rad
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

vehicle_clustering_results = []

for taxiid, vehicle_data in dedicated_charges_df.groupby('taxiid'):
    n_events = len(vehicle_data)

    if n_events < 2:
        continue

    # ========== 1. Spatial Clustering Analysis ==========
    coords = vehicle_data[['start_lon', 'start_lat']].values
    coords_rad = np.radians(coords)
    eps_rad = MERGE_DISTANCE_M / 6371000

    dbscan = DBSCAN(eps=eps_rad, min_samples=1, algorithm='ball_tree', metric='haversine')
    cluster_labels = dbscan.fit_predict(coords_rad)

    # Count cluster sizes
    unique_labels = set(cluster_labels)
    if -1 in unique_labels:
        unique_labels.remove(-1)

    if len(unique_labels) == 0:
        spatial_score = 0
        n_main_clusters = 0
    else:
        cluster_counts = pd.Series(cluster_labels).value_counts()
        if -1 in cluster_counts.index:
            cluster_counts = cluster_counts.drop(-1)

        # Find main clusters covering 80% of events
        cluster_counts_sorted = cluster_counts.sort_values(ascending=False)
        cumulative_ratio = cluster_counts_sorted.cumsum() / n_events

        main_clusters = cluster_counts_sorted[cumulative_ratio <= SPATIAL_THRESHOLD_RATIO]
        if len(main_clusters) == 0:
            main_clusters = cluster_counts_sorted.head(1)

        n_main_clusters = len(main_clusters)
        spatial_score = main_clusters.sum() / n_events

    # ========== 2. Temporal Clustering Analysis ==========
    hour_counts = vehicle_data['hour'].value_counts()

    # Find main time slots covering 80% of events
    hour_counts_sorted = hour_counts.sort_values(ascending=False)
    cumulative_ratio_time = hour_counts_sorted.cumsum() / n_events

    main_hours = hour_counts_sorted[cumulative_ratio_time <= TIME_THRESHOLD_RATIO]
    if len(main_hours) == 0:
        main_hours = hour_counts_sorted.head(1)

    # Merge consecutive hours into time slots
    main_hour_set = set(main_hours.index)
    time_slots = []
    current_slot = None

    for hour in sorted(main_hour_set):
        if current_slot is None:
            current_slot = [hour, hour]
        elif hour == current_slot[1] + 1:
            current_slot[1] = hour
        else:
            time_slots.append(tuple(current_slot))
            current_slot = [hour, hour]
    if current_slot:
        time_slots.append(tuple(current_slot))

    n_main_time_slots = len(time_slots)
    time_score = main_hours.sum() / n_events

    # Combined score
    combined_score = (spatial_score * 0.5) + (time_score * 0.5)

    vehicle_clustering_results.append({
        'taxiid': taxiid,
        'n_events': n_events,
        'n_main_clusters': n_main_clusters,
        'spatial_score': spatial_score,
        'n_main_time_slots': n_main_time_slots,
        'time_score': time_score,
        'combined_score': combined_score
    })

results_df = pd.DataFrame(vehicle_clustering_results)

# ========== 1. Filtering by Spatial and Temporal Clustering Scores ==========
print("=" * 60)
print("Filtering by Spatial and Temporal Clustering Scores")
print("=" * 60)

# Set filtering thresholds
spatial_threshold = results_df['spatial_score'].quantile(0.1)  # 10th percentile
time_threshold = results_df['time_score'].quantile(0.1)        # 10th percentile

print(f"\nFiltering thresholds:")
print(f"  Spatial score >= {spatial_threshold:.2%}")
print(f"  Time score >= {time_threshold:.2%}")

# Filter: keep vehicles with high spatial and temporal clustering
filtered_df = results_df[
    (results_df['spatial_score'] >= spatial_threshold) &
    (results_df['time_score'] >= time_threshold)
].copy()

print(f"\nBefore filtering:")
print(f"  Total vehicles: {len(results_df)}")
print(f"  Mean spatial score: {results_df['spatial_score'].mean():.2%}")
print(f"  Mean time score: {results_df['time_score'].mean():.2%}")
print(f"  Mean combined score: {results_df['combined_score'].mean():.2%}")

print(f"\nAfter filtering:")
print(f"  Remaining vehicles: {len(filtered_df)} ({len(filtered_df)/len(results_df)*100:.2f}%)")
print(f"  Mean spatial score: {filtered_df['spatial_score'].mean():.2%}")
print(f"  Mean time score: {filtered_df['time_score'].mean():.2%}")
print(f"  Mean combined score: {filtered_df['combined_score'].mean():.2%}")

# ========== 2. Distribution Statistics After Filtering ==========
print(f"\n" + "=" * 60)
print("Distribution Statistics After Filtering")
print("=" * 60)

# Cluster count distribution
print(f"\nNumber of Main Clusters Distribution:")
cluster_dist = filtered_df['n_main_clusters'].value_counts().sort_index()
for n_clusters, count in cluster_dist.items():
    print(f"  {n_clusters} clusters: {count} vehicles ({count/len(filtered_df)*100:.2f}%)")

# Time slot count distribution
print(f"\nNumber of Main Time Slots Distribution:")
time_dist = filtered_df['n_main_time_slots'].value_counts().sort_index()
for n_slots, count in time_dist.items():
    print(f"  {n_slots} time slots: {count} vehicles ({count/len(filtered_df)*100:.2f}%)")

# ========== 3. Visualize Filtered Distributions ==========
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 3.1 Cluster count distribution
axes[0, 0].hist(filtered_df['n_main_clusters'],
                bins=range(int(filtered_df['n_main_clusters'].max())+2),
                edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_xlabel('Number of Main Clusters', fontsize=11)
axes[0, 0].set_ylabel('Number of Vehicles', fontsize=11)
axes[0, 0].set_title('Distribution of Main Clusters (After Filtering)', fontsize=12)
axes[0, 0].grid(axis='y', alpha=0.3)
axes[0, 0].axvline(filtered_df['n_main_clusters'].mean(), color='red', linestyle='--',
                   label=f'Mean: {filtered_df["n_main_clusters"].mean():.2f}')
axes[0, 0].legend()

# 3.2 Time slot count distribution
axes[0, 1].hist(filtered_df['n_main_time_slots'],
                bins=range(int(filtered_df['n_main_time_slots'].max())+2),
                edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_xlabel('Number of Main Time Slots', fontsize=11)
axes[0, 1].set_ylabel('Number of Vehicles', fontsize=11)
axes[0, 1].set_title('Distribution of Main Time Slots (After Filtering)', fontsize=12)
axes[0, 1].grid(axis='y', alpha=0.3)
axes[0, 1].axvline(filtered_df['n_main_time_slots'].mean(), color='red', linestyle='--',
                   label=f'Mean: {filtered_df["n_main_time_slots"].mean():.2f}')
axes[0, 1].legend()

# 3.3 Spatial clustering score distribution
axes[1, 0].hist(filtered_df['spatial_score'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_xlabel('Spatial Clustering Score', fontsize=11)
axes[1, 0].set_ylabel('Number of Vehicles', fontsize=11)
axes[1, 0].set_title('Distribution of Spatial Clustering Score (After Filtering)', fontsize=12)
axes[1, 0].grid(axis='y', alpha=0.3)
axes[1, 0].axvline(filtered_df['spatial_score'].mean(), color='red', linestyle='--',
                   label=f'Mean: {filtered_df["spatial_score"].mean():.2%}')
axes[1, 0].legend()

# 3.4 Temporal clustering score distribution
axes[1, 1].hist(filtered_df['time_score'], bins=30, edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].set_xlabel('Temporal Clustering Score', fontsize=11)
axes[1, 1].set_ylabel('Number of Vehicles', fontsize=11)
axes[1, 1].set_title('Distribution of Temporal Clustering Score (After Filtering)', fontsize=12)
axes[1, 1].grid(axis='y', alpha=0.3)
axes[1, 1].axvline(filtered_df['time_score'].mean(), color='red', linestyle='--',
                   label=f'Mean: {filtered_df["time_score"].mean():.2%}')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# ========== 4. Scatter Plot: Clusters vs Time Slots ==========
fig, ax = plt.subplots(1, 1, figsize=(10, 7))

scatter = ax.scatter(filtered_df['n_main_clusters'], filtered_df['n_main_time_slots'],
                    c=filtered_df['combined_score'], cmap='RdYlGn',
                    s=80, alpha=0.6, edgecolors='black', linewidth=0.5)
ax.set_xlabel('Number of Main Clusters', fontsize=12)
ax.set_ylabel('Number of Main Time Slots', fontsize=12)
ax.set_title('Clusters vs Time Slots (Color = Combined Clustering Score)', fontsize=13)
plt.colorbar(scatter, ax=ax, label='Combined Clustering Score')
ax.grid(True, alpha=0.3)

# Add statistics text box
stats_text = f'Total Vehicles: {len(filtered_df)}\n'
stats_text += f'Mean Clusters: {filtered_df["n_main_clusters"].mean():.2f}\n'
stats_text += f'Mean Time Slots: {filtered_df["n_main_time_slots"].mean():.2f}\n'
stats_text += f'Mean Combined Score: {filtered_df["combined_score"].mean():.2%}'

ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
        fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("Filtering completed. Visualizations shown above.")
print("=" * 60)

In [ ]:
import folium
from folium.plugins import TimestampedGeoJson
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.colors as mcolors

# ========== Visualization 3: Trajectory Visualization for 5 Random Vehicles ==========
print("\n" + "="*60)
print("Visualization 3: Trajectory Visualization for 5 Random Vehicles")
print("="*60)

# Select 5 random vehicles
np.random.seed(50)
selected_vehicles = np.random.choice(filterids, size=min(3, len(filterids)), replace=False)

# Load all charging events to identify charging periods
all_charging_events = pd.read_parquet('output_v4/battery_trace_p.parquet')
all_charging_events = all_charging_events[all_charging_events['taxiid'].isin(filterids)].copy()
all_charging_events['start_time'] = pd.to_datetime(all_charging_events['start_time'])
all_charging_events['end_time'] = pd.to_datetime(all_charging_events['end_time'])

# Shenzhen center coordinates (approximate)
SHENZHEN_CENTER = [22.5431, 114.0579]

# Color palette for different days
day_colors = ['blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan', 'magenta']

for vehicle_id in selected_vehicles:
    print(f"\nProcessing vehicle: {vehicle_id}")

    # Load trajectory data
    track_path = Path(TRACKS_DIR) / f'taxiid={vehicle_id}'
    track_files = list(track_path.glob('*.parquet'))

    if not track_files:
        print(f"  No trajectory data found for {vehicle_id}")
        continue

    # Read trajectory
    track = pd.concat([pd.read_parquet(f) for f in track_files], ignore_index=True)
    track = track.sort_values('time').reset_index(drop=True)
    track['time'] = pd.to_datetime(track['time'])
    track['date'] = track['time'].dt.date  # Extract date

    # Get charging events for this vehicle
    vehicle_charges = all_charging_events[all_charging_events['taxiid'] == vehicle_id].copy()
    vehicle_charges['date'] = vehicle_charges['start_time'].dt.date  # Extract date

    # Mark charging periods in trajectory
    track['is_charging'] = False
    track['charge_type'] = None

    for _, charge in vehicle_charges.iterrows():
        mask = (track['time'] >= charge['start_time']) & (track['time'] <= charge['end_time'])
        track.loc[mask, 'is_charging'] = True
        track.loc[mask, 'charge_type'] = charge['charge_type']

    # Get unique dates
    unique_dates = sorted(track['date'].unique())
    print(f"  Dates in trajectory: {unique_dates[:10]}...")  # Show first 10 dates

    # Calculate map center
    if len(track) > 0:
        center_lat = track['lat'].mean()
        center_lon = track['lon'].mean()
    else:
        center_lat, center_lon = SHENZHEN_CENTER

    # Create folium map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles='OpenStreetMap'
    )

    # Plot trajectory by day
    for day_idx, date in enumerate(unique_dates):
        day_track = track[track['date'] == date].copy()

        if len(day_track) == 0:
            continue

        # Separate normal driving and charging for this day
        day_normal = day_track[~day_track['is_charging']].copy()
        day_charging = day_track[day_track['is_charging']].copy()

        # Get color for this day (cycle through colors)
        color = day_colors[day_idx % len(day_colors)]

        # Plot normal driving trajectory for this day
        if len(day_normal) > 1:
            # Sample points if too many (for performance)
            if len(day_normal) > 5000:
                day_normal_sample = day_normal.iloc[::len(day_normal)//5000]
            else:
                day_normal_sample = day_normal

            normal_coords = [[row['lat'], row['lon']] for _, row in day_normal_sample.iterrows()]

            folium.PolyLine(
                normal_coords,
                color=color,
                weight=2,
                opacity=0.6,
                popup=f'Normal Driving - {date}<br>{len(day_normal)} points',
                tooltip=f'Normal Driving - {date}'
            ).add_to(m)

        # Plot charging locations for this day
        if len(day_charging) > 0:
            # Separate public and dedicated charging
            day_public = day_charging[day_charging['charge_type'] == 'public'].copy()
            day_dedicated = day_charging[day_charging['charge_type'] == 'dedicated_station'].copy()

            # Group consecutive charging points
            def group_consecutive_points(charging_df):
                if len(charging_df) == 0:
                    return []

                charging_groups = []
                current_group = []
                prev_idx = None

                for idx, row in charging_df.iterrows():
                    if prev_idx is None or idx != prev_idx + 1:
                        if current_group:
                            charging_groups.append(current_group)
                        current_group = [row]
                    else:
                        current_group.append(row)
                    prev_idx = idx

                if current_group:
                    charging_groups.append(current_group)

                return charging_groups

            # Plot public charging (red circles)
            public_groups = group_consecutive_points(day_public)
            for group_idx, group in enumerate(public_groups):
                group_df = pd.DataFrame(group)
                group_lat = group_df['lat'].mean()
                group_lon = group_df['lon'].mean()
                start_time = group_df['time'].min()
                end_time = group_df['time'].max()
                duration_min = (end_time - start_time).total_seconds() / 60

                folium.CircleMarker(
                    location=[group_lat, group_lon],
                    radius=8,
                    popup=f'Public Charging - {date}<br>'
                          f'Event {group_idx+1}<br>'
                          f'Start: {start_time.strftime("%Y-%m-%d %H:%M:%S")}<br>'
                          f'End: {end_time.strftime("%Y-%m-%d %H:%M:%S")}<br>'
                          f'Duration: {duration_min:.1f} minutes',
                    tooltip=f'Public Charging {group_idx+1} - {date}',
                    color='red',
                    fill=True,
                    fillColor='red',
                    fillOpacity=0.8,
                    weight=2
                ).add_to(m)

                folium.Circle(
                    location=[group_lat, group_lon],
                    radius=100,
                    popup=f'Public Charging Area - {date}',
                    color='red',
                    fill=True,
                    fillColor='red',
                    fillOpacity=0.2,
                    weight=1
                ).add_to(m)

            # Plot dedicated charging (orange squares)
            dedicated_groups = group_consecutive_points(day_dedicated)
            for group_idx, group in enumerate(dedicated_groups):
                group_df = pd.DataFrame(group)
                group_lat = group_df['lat'].mean()
                group_lon = group_df['lon'].mean()
                start_time = group_df['time'].min()
                end_time = group_df['time'].max()
                duration_hours = (end_time - start_time).total_seconds() / 3600

                folium.Marker(
                    location=[group_lat, group_lon],
                    popup=f'Dedicated Charging - {date}<br>'
                          f'Event {group_idx+1}<br>'
                          f'Start: {start_time.strftime("%Y-%m-%d %H:%M:%S")}<br>'
                          f'End: {end_time.strftime("%Y-%m-%d %H:%M:%S")}<br>'
                          f'Duration: {duration_hours:.2f} hours',
                    tooltip=f'Dedicated Charging {group_idx+1} - {date}',
                    icon=folium.Icon(color='orange', icon='plug', prefix='fa')
                ).add_to(m)

                folium.Circle(
                    location=[group_lat, group_lon],
                    radius=200,
                    popup=f'Dedicated Charging Area - {date}',
                    color='orange',
                    fill=True,
                    fillColor='orange',
                    fillOpacity=0.15,
                    weight=2,
                    dashArray='5, 5'  # Dashed line
                ).add_to(m)

    # Add start and end markers
    if len(track) > 0:
        # Start point
        folium.Marker(
            location=[track.iloc[0]['lat'], track.iloc[0]['lon']],
            popup=f'Start Point<br>Time: {track.iloc[0]["time"].strftime("%Y-%m-%d %H:%M:%S")}',
            tooltip='Start',
            icon=folium.Icon(color='green', icon='play', prefix='fa')
        ).add_to(m)

        # End point
        folium.Marker(
            location=[track.iloc[-1]['lat'], track.iloc[-1]['lon']],
            popup=f'End Point<br>Time: {track.iloc[-1]["time"].strftime("%Y-%m-%d %H:%M:%S")}',
            tooltip='End',
            icon=folium.Icon(color='darkred', icon='stop', prefix='fa')
        ).add_to(m)

    # Add legend with date colors
    legend_html = f'''
    <div style="position: fixed;
                bottom: 50px; left: 50px; width: 250px; height: auto;
                background-color: white; border:2px solid grey; z-index:9999;
                font-size:12px; padding: 10px; max-height: 400px; overflow-y: auto;">
    <h4>Legend - Vehicle {vehicle_id}</h4>
    <p><i class="fa fa-circle" style="color:blue"></i> Normal Driving (by day)</p>
    <p><i class="fa fa-circle" style="color:red"></i> Public Charging</p>
    <p><i class="fa fa-plug" style="color:orange"></i> Dedicated Charging</p>
    <p><i class="fa fa-play" style="color:green"></i> Start</p>
    <p><i class="fa fa-stop" style="color:darkred"></i> End</p>
    <hr>
    <h5>Days:</h5>
    '''

    for day_idx, date in enumerate(unique_dates[:15]):  # Show first 15 days
        color = day_colors[day_idx % len(day_colors)]
        legend_html += f'<p><span style="color:{color};">■</span> {date}</p>'

    if len(unique_dates) > 15:
        legend_html += f'<p>... and {len(unique_dates) - 15} more days</p>'

    legend_html += '</div>'
    m.get_root().html.add_child(folium.Element(legend_html))

    # Display map
    display(m)

    # Print statistics
    print(f"  Total trajectory points: {len(track)}")
    print(f"  Normal driving points: {len(track[~track['is_charging']])}")
    print(f"  Charging points: {len(track[track['is_charging']])}")
    print(f"  Public charging events: {len(vehicle_charges[vehicle_charges['charge_type'] == 'public'])}")
    print(f"  Dedicated charging events: {len(vehicle_charges[vehicle_charges['charge_type'] == 'dedicated_station'])}")
    print(f"  Total charging events: {len(vehicle_charges)}")
    print(f"  Number of days: {len(unique_dates)}")
    print(f"  Map center: [{center_lat:.4f}, {center_lon:.4f}]")

print("\n" + "="*60)
print("All visualizations completed!")
print("="*60)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

# Load dedicated charging events
dedicated_charges_df = pd.read_parquet('output_v4/private_gap_charging_events_p.parquet')
dedicated_charges_df = dedicated_charges_df[dedicated_charges_df['taxiid'].isin(filterids)].copy()

MERGE_DISTANCE_M = 500  # 500 meters

# Calculate average events per cluster point for each vehicle
avg_events_per_point = []

for taxiid, vehicle_data in dedicated_charges_df.groupby('taxiid'):
    coords = vehicle_data[['start_lon', 'start_lat']].values
    coords_rad = np.radians(coords)
    eps_rad = MERGE_DISTANCE_M / 6371000

    # DBSCAN clustering
    dbscan = DBSCAN(eps=eps_rad, min_samples=1, algorithm='ball_tree', metric='haversine')
    cluster_labels = dbscan.fit_predict(coords_rad)

    # Count unique clusters (excluding noise)
    unique_labels = set(cluster_labels)
    if -1 in unique_labels:
        unique_labels.remove(-1)

    n_clusters = len(unique_labels)
    n_events = len(vehicle_data)

    if n_clusters > 0:
        avg_events = n_events / n_clusters
        avg_events_per_point.append(avg_events)

# Plot distribution
plt.figure(figsize=(10, 6))
plt.hist(avg_events_per_point, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
plt.xlabel('Average Number of Charging Events per Cluster Point', fontsize=12)
plt.ylabel('Number of Vehicles', fontsize=12)
#plt.title('Distribution of Average Charging Events per Cluster Point\n(Dedicated Charging, 500m clustering)', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# Add mean line
mean_value = np.mean(avg_events_per_point)
median_value = np.median(avg_events_per_point)
plt.axvline(mean_value, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_value:.2f}')
plt.axvline(median_value, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_value:.2f}')
plt.legend(fontsize=10)

plt.tight_layout()
plt.show()

# Print statistics
print(f"Total vehicles: {len(avg_events_per_point)}")
print(f"Mean events per point: {mean_value:.2f}")
print(f"Median events per point: {median_value:.2f}")
print(f"Min events per point: {np.min(avg_events_per_point):.2f}")
print(f"Max events per point: {np.max(avg_events_per_point):.2f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import folium
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import random

# 设置随机种子以便结果可复现
random.seed(45)
np.random.seed(45)

# 1. 加载dedicated charging events数据
print("正在加载数据...")
dedicated_charges_df = pd.read_parquet('output_v4/centralized_charging_events_p.parquet')
print(f"总共有 {len(dedicated_charges_df)} 个dedicated charging events")

# 检查数据列
print(f"\n数据列: {dedicated_charges_df.columns.tolist()}")
#print(f"\n数据示例:\n{dedicated_charges_df[['stay_lon', 'stay_lat']].head()}")

# 2. 进行5次采样，每次随机抽取3万个
n_samples = 20000
n_iterations = 6

# 确保有足够的数据
if len(dedicated_charges_df) < n_samples:
    print(f"警告: 数据量 ({len(dedicated_charges_df)}) 少于采样数量 ({n_samples})")
    n_samples = len(dedicated_charges_df)

# 创建地图（深圳中心坐标）
shenzhen_center = [22.5431, 114.0579]
m = folium.Map(
    location=shenzhen_center,
    zoom_start=11,
    tiles='OpenStreetMap'
)

# 存储所有聚类的统计信息
all_stats = []

# 3. 对每次采样进行DBSCAN聚类
for iteration in range(n_iterations):
    print(f"\n{'='*60}")
    print(f"第 {iteration + 1} 次采样和聚类")
    print(f"{'='*60}")

    # 随机采样
    sampled_df = dedicated_charges_df.sample(n=n_samples, random_state=45+iteration*100)
    print(f"采样了 {len(sampled_df)} 个事件")

    # 提取位置坐标
    coords = sampled_df[['start_lon', 'start_lat']].values

    # DBSCAN聚类
    # eps: 邻域半径（度），约500米 ≈ 0.0045度
    # min_samples: 最小样本数
    eps = 0.0045  # 约500米
    min_samples = 5

    dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='haversine')
    # 注意：DBSCAN的haversine需要弧度，且输入格式为 [lat, lon]
    coords_rad = np.radians(coords[:, [1, 0]])  # 转换为 [lat, lon] 并转为弧度
    labels = dbscan.fit_predict(coords_rad)

    # 统计信息
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)

    print(f"聚类结果:")
    print(f"  - 聚类数量: {n_clusters}")
    print(f"  - 噪声点数量: {n_noise}")
    print(f"  - 聚类点数量: {len(labels) - n_noise}")

    # 计算每个聚类的中心点
    cluster_centers = []
    cluster_sizes = []

    for cluster_id in set(labels):
        if cluster_id == -1:  # 跳过噪声点
            continue

        cluster_mask = labels == cluster_id
        cluster_points = coords[cluster_mask]

        # 计算聚类中心（均值）
        center_lon = cluster_points[:, 0].mean()
        center_lat = cluster_points[:, 1].mean()
        cluster_size = len(cluster_points)

        cluster_centers.append({
            'lon': center_lon,
            'lat': center_lat,
            'size': cluster_size,
            'iteration': iteration + 1
        })
        cluster_sizes.append(cluster_size)

    # 存储统计信息
    all_stats.append({
        'iteration': iteration + 1,
        'n_clusters': n_clusters,
        'n_noise': n_noise,
        'cluster_sizes': cluster_sizes,
        'avg_cluster_size': np.mean(cluster_sizes) if cluster_sizes else 0,
        'max_cluster_size': np.max(cluster_sizes) if cluster_sizes else 0,
        'min_cluster_size': np.min(cluster_sizes) if cluster_sizes else 0
    })



In [ ]:
# 在地图上绘制聚类中心点
# 使用不同颜色区分多次采样
colors = ['red', 'blue', 'green', 'yellow', 'pink', 'magenta']
color = colors[iteration]

for center in cluster_centers:
    # 根据聚类大小设置标记大小
    radius = min(10 + center['size'] / 100, 30)

    folium.CircleMarker(
        location=[center['lat'], center['lon']],
        radius=radius,
        popup=f"第{center['iteration']}次采样<br>聚类大小: {center['size']}<br>位置: ({center['lat']:.6f}, {center['lon']:.6f})",
        tooltip=f"聚类 {center['size']} 个点",
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.6,
        weight=2
    ).add_to(m)

print(f"已在地图上标记 {len(cluster_centers)} 个聚类中心")

# 4. 打印总体统计信息（在循环外）
print(f"\n{'='*60}")
print("总体统计信息")
print(f"{'='*60}")

for stats in all_stats:
    print(f"\n第 {stats['iteration']} 次采样:")
    print(f"  聚类数量: {stats['n_clusters']}")
    print(f"  噪声点数量: {stats['n_noise']}")
    if stats['cluster_sizes']:
        print(f"  平均聚类大小: {stats['avg_cluster_size']:.1f}")
        print(f"  最大聚类大小: {stats['max_cluster_size']}")
        print(f"  最小聚类大小: {stats['min_cluster_size']}")

# 计算多次采样的平均统计
avg_n_clusters = np.mean([s['n_clusters'] for s in all_stats])
avg_n_noise = np.mean([s['n_noise'] for s in all_stats])
avg_cluster_size = np.mean([s['avg_cluster_size'] for s in all_stats])

# 5. 添加图例（在循环外，只执行一次）
colors = ['red', 'blue', 'green', 'yellow', 'pink', 'cyan']
n_iterations = len(all_stats)  # 获取实际采样次数

# 动态生成图例HTML
legend_items = []
for i in range(n_iterations):
    color = colors[i % len(colors)]  # 如果采样次数超过颜色数量，循环使用
    legend_items.append(f'<p><span style="color:{color}">●</span> 第{i+1}次采样</p>')

legend_html = f'''
<div style="position: fixed;
     bottom: 50px; left: 50px; width: 200px; height: auto;
     background-color: white; border:2px solid grey; z-index:9999;
     font-size:14px; padding: 10px">
     <h4>图例</h4>
     {''.join(legend_items)}
     <p>圆圈大小表示聚类大小</p>
</div>
'''

# 只在所有循环结束后添加一次图例
m.get_root().html.add_child(folium.Element(legend_html))

# # 6. 保存地图
# map_filename = 'dedicated_charging_clusters_map.html'
# m.save(map_filename)
# print(f"\n地图已保存到: {map_filename}")

# 7. 显示地图（在Jupyter中）
m

In [ ]:

# ========== 读取数据 ==========
char_queue_file = CHAR_QUEUE_FILE  # 根据实际路径调整
if os.path.exists(char_queue_file):
    cq_df = pd.read_parquet(char_queue_file)
else:
    # 尝试其他可能的路径
    CHAR_QUEUE_FILE = f'{OUTPUT_DIR}/char_queue_test.parquet'
    if os.path.exists(CHAR_QUEUE_FILE):
        cq_df = pd.read_parquet(CHAR_QUEUE_FILE)
    else:
        print(f"Error: Cannot find char_queue.parquet file")
        cq_df = pd.DataFrame()

if not cq_df.empty:
    cq_df = cq_df[cq_df['taxiid'].isin(filterids)]
    # 确保时间列是datetime类型
    if 'arrive_time' in cq_df.columns:
        cq_df['arrive_time'] = pd.to_datetime(cq_df['arrive_time'])
        cq_df['date'] = cq_df['arrive_time'].dt.date

    print("=" * 60)
    print("Public Charging Statistics")
    print("=" * 60)

    # 1. 每车每日平均充电次数
    if 'taxiid' in cq_df.columns and 'date' in cq_df.columns:
        daily_charges = cq_df.groupby(['taxiid', 'date']).size().reset_index(name='charges_per_day')
        avg_charges_per_day = daily_charges.groupby('taxiid')['charges_per_day'].mean()

        print(f"\n1. Average Charging Frequency per Vehicle per Day:")
        print(f"   Total vehicles: {len(avg_charges_per_day)}")
        print(f"   Mean: {avg_charges_per_day.mean():.2f} times/day")
        print(f"   Median: {avg_charges_per_day.median():.2f} times/day")
        print(f"   Min: {avg_charges_per_day.min():.2f} times/day")
        print(f"   Max: {avg_charges_per_day.max():.2f} times/day")
        print(f"   Std: {avg_charges_per_day.std():.2f} times/day")

    # 2. 在站停留时间分布（charge_dur）
    if 'charge_dur' in cq_df.columns:
        # 转换为分钟
        charge_dur_minutes = (cq_df['charge_dur'] + cq_df['wait_dur'])/ 60

        print(f"\n2. Charging Duration Distribution (at station):")
        print(f"   Total events: {len(charge_dur_minutes)}")
        print(f"   Mean: {charge_dur_minutes.mean():.2f} minutes")
        print(f"   Median: {charge_dur_minutes.median():.2f} minutes")
        print(f"   Min: {charge_dur_minutes.min():.2f} minutes")
        print(f"   Max: {charge_dur_minutes.max():.2f} minutes")
        print(f"   Std: {charge_dur_minutes.std():.2f} minutes")
        print(f"   10th percentile: {charge_dur_minutes.quantile(0.10):.2f} minutes")
        print(f"   25th percentile: {charge_dur_minutes.quantile(0.25):.2f} minutes")
        print(f"   75th percentile: {charge_dur_minutes.quantile(0.75):.2f} minutes")
        print(f"   95th percentile: {charge_dur_minutes.quantile(0.95):.2f} minutes")

    # 3. 等待时间分布（wait_dur）
    if 'wait_dur' in cq_df.columns:
        # 转换为分钟
        wait_dur_minutes = cq_df['wait_dur'] / 60

        print(f"\n3. Waiting Time Distribution:")
        print(f"   Total events: {len(wait_dur_minutes)}")
        print(f"   Events with wait > 0: {(wait_dur_minutes > 0).sum()} ({(wait_dur_minutes > 0).sum()/len(wait_dur_minutes)*100:.2f}%)")
        print(f"   Mean (all): {wait_dur_minutes.mean():.2f} minutes")
        print(f"   Mean (wait > 0): {wait_dur_minutes[wait_dur_minutes > 0].mean():.2f} minutes" if (wait_dur_minutes > 0).sum() > 0 else "   Mean (wait > 0): N/A")
        print(f"   Median: {wait_dur_minutes.median():.2f} minutes")
        print(f"   Max: {wait_dur_minutes.max():.2f} minutes")
        print(f"   5th percentile: {wait_dur_minutes.quantile(0.05):.2f} minutes")
        print(f"   75th percentile: {wait_dur_minutes.quantile(0.75):.2f} minutes")
        print(f"   90th percentile: {wait_dur_minutes.quantile(0.90):.2f} minutes")
        print(f"   95th percentile: {wait_dur_minutes.quantile(0.95):.2f} minutes")

    # 4. 可视化
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # 4.1 每车每日平均充电次数分布
    if 'taxiid' in cq_df.columns and 'date' in cq_df.columns:
        axes[0, 0].hist(avg_charges_per_day.values, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
        axes[0, 0].set_xlabel('Average Charges per Day')
        axes[0, 0].set_ylabel('Number of Vehicles')
        axes[0, 0].set_title('Distribution of Average Daily Charging Frequency')
        axes[0, 0].grid(axis='y', alpha=0.3)
        axes[0, 0].axvline(avg_charges_per_day.mean(), color='red', linestyle='--', label=f'Mean: {avg_charges_per_day.mean():.2f}')
        axes[0, 0].legend()

    # 4.2 充电时长分布
    if 'charge_dur' in cq_df.columns:
        charge_dur_minutes = cq_df['charge_dur'] / 60
        # 限制显示范围到合理值（例如0-180分钟）
        charge_dur_filtered = charge_dur_minutes[charge_dur_minutes <= 180]
        axes[0, 1].hist(charge_dur_filtered.values, bins=50, color='green', alpha=0.7, edgecolor='black')
        axes[0, 1].set_xlabel('Charging Duration (minutes)')
        axes[0, 1].set_ylabel('Number of Events')
        axes[0, 1].set_title('Distribution of Charging Duration at Station')
        axes[0, 1].grid(axis='y', alpha=0.3)
        axes[0, 1].axvline(charge_dur_minutes.mean(), color='red', linestyle='--', label=f'Mean: {charge_dur_minutes.mean():.2f} min')
        axes[0, 1].legend()

    # 4.3 等待时间分布（所有事件）
    if 'wait_dur' in cq_df.columns:
        wait_dur_minutes = cq_df['wait_dur'] / 60
        # 限制显示范围（例如0-120分钟）
        wait_dur_filtered = wait_dur_minutes[wait_dur_minutes <= 120]
        axes[1, 0].hist(wait_dur_filtered.values, bins=50, color='orange', alpha=0.7, edgecolor='black')
        axes[1, 0].set_xlabel('Waiting Time (minutes)')
        axes[1, 0].set_ylabel('Number of Events')
        axes[1, 0].set_title('Distribution of Waiting Time (All Events)')
        axes[1, 0].grid(axis='y', alpha=0.3)
        axes[1, 0].axvline(wait_dur_minutes.mean(), color='red', linestyle='--', label=f'Mean: {wait_dur_minutes.mean():.2f} min')
        axes[1, 0].legend()

    # 4.4 等待时间分布（仅等待时间>0的事件）
    if 'wait_dur' in cq_df.columns:
        wait_dur_minutes = cq_df['wait_dur'] / 60
        wait_positive = wait_dur_minutes[wait_dur_minutes > 0]
        if len(wait_positive) > 0:
            wait_positive_filtered = wait_positive[wait_positive <= 120]
            axes[1, 1].hist(wait_positive_filtered.values, bins=50, color='coral', alpha=0.7, edgecolor='black')
            axes[1, 1].set_xlabel('Waiting Time (minutes)')
            axes[1, 1].set_ylabel('Number of Events')
            axes[1, 1].set_title(f'Distribution of Waiting Time (Wait > 0, n={len(wait_positive)})')
            axes[1, 1].grid(axis='y', alpha=0.3)
            axes[1, 1].axvline(wait_positive.mean(), color='red', linestyle='--', label=f'Mean: {wait_positive.mean():.2f} min')
            axes[1, 1].legend()
        else:
            axes[1, 1].text(0.5, 0.5, 'No events with wait > 0', ha='center', va='center', transform=axes[1, 1].transAxes)
            axes[1, 1].set_title('Distribution of Waiting Time (Wait > 0)')

    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 60)

else:
    print("Error: char_queue DataFrame is empty or file not found")

In [ ]:
# 基本统计
if 'char_queue_df' in locals() and not char_queue_df.empty:
    print("=" * 60)
    print("排队和充电统计")
    print("=" * 60)

    print(f"\n总充电事件数: {len(char_queue_df):,}")
    print(f"涉及车辆数: {char_queue_df['taxiid'].nunique():,}")
    print(f"涉及充电站数: {char_queue_df['nearest_station_id'].nunique():,}")

    print(f"\n平均等待时间: {char_queue_df['wait_dur'].mean():.2f} 秒 ({char_queue_df['wait_dur'].mean()/60:.2f} 分钟)")
    print(f"中位数等待时间: {char_queue_df['wait_dur'].median():.2f} 秒 ({char_queue_df['wait_dur'].median()/60:.2f} 分钟)")
    print(f"95%分位数等待时间: {char_queue_df['wait_dur'].quantile(0.95):.2f} 秒 ({char_queue_df['wait_dur'].quantile(0.95)/60:.2f} 分钟)")

    print(f"\n放弃率: {char_queue_df['giveup'].mean():.2%}")
    print(f"放弃事件数: {char_queue_df['giveup'].sum():,}")

    successful_charges = char_queue_df[char_queue_df['giveup'] == False]
    if not successful_charges.empty:
        print(f"\n成功充电事件数: {len(successful_charges):,}")
        print(f"平均充电时长: {successful_charges['charge_dur'].mean()/60:.2f} 分钟")
        print(f"中位数充电时长: {successful_charges['charge_dur'].median()/60:.2f} 分钟")

    print("=" * 60)


In [ ]:
# 每日充电次数统计
if 'char_queue_df' in locals() and not char_queue_df.empty:
    char_queue_df['charge_date'] = char_queue_df['arrive_time'].dt.date

    # 每辆车每天的成功充电次数
    daily_charges = char_queue_df[char_queue_df['giveup'] == False].groupby(['taxiid', 'charge_date']).size().reset_index(name='daily_charge_count')

    # 每辆车的平均每日充电次数
    avg_daily_per_taxi = daily_charges.groupby('taxiid')['daily_charge_count'].mean().reset_index(name='avg_daily_charges')

    overall_avg = avg_daily_per_taxi['avg_daily_charges'].mean()

    print(f"\n平均每日充电次数（每车）: {overall_avg:.2f} 次/天")
    print(f"中位数每日充电次数: {avg_daily_per_taxi['avg_daily_charges'].median():.2f} 次/天")


In [ ]:
# SOC统计
if os.path.exists(BATTERY_TRACE_FILE):
    battery_trace = pd.read_parquet(BATTERY_TRACE_FILE)

    print("\n" + "=" * 60)
    print("SOC统计")
    print("=" * 60)

    print(f"\n充电前SOC统计:")
    print(f"  平均: {battery_trace['SOC_before_pct'].mean():.2f}%")
    print(f"  中位数: {battery_trace['SOC_before_pct'].median():.2f}%")
    print(f"  最小值: {battery_trace['SOC_before_pct'].min():.2f}%")

    low_soc = (battery_trace['SOC_before_pct'] < 20).sum()
    print(f"\n低电量充电次数 (SOC < 20%): {low_soc:,} ({low_soc/len(battery_trace)*100:.2f}%)")

    print(f"\n充电后SOC统计:")
    print(f"  平均: {battery_trace['SOC_after_pct'].mean():.2f}%")
    print(f"  中位数: {battery_trace['SOC_after_pct'].median():.2f}%")
    print(f"  最大值: {battery_trace['SOC_after_pct'].max():.2f}%")

    print("=" * 60)


### 7.2 可视化


In [ ]:
# 等待时间分布
if 'char_queue_df' in locals() and not char_queue_df.empty:
    waiting_events = char_queue_df[char_queue_df['wait_dur'] > 0.1]  # 只显示有等待的

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 等待时间直方图
    axes[0].hist(waiting_events['wait_dur'] / 60, bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('等待时间 (分钟)')
    axes[0].set_ylabel('事件数')
    axes[0].set_title('等待时间分布')
    axes[0].set_xlim(0, 120)  # 最多显示2小时

    # 总停留时长分布
    char_queue_df['stay_duration_min'] = (
        (char_queue_df['leave_time'] - char_queue_df['arrive_time']).dt.total_seconds() / 60
    )
    axes[1].hist(char_queue_df['stay_duration_min'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('总停留时长 (分钟)')
    axes[1].set_ylabel('事件数')
    axes[1].set_title('总停留时长分布')
    axes[1].set_xlim(0, 180)  # 最多显示3小时

    plt.tight_layout()
    plt.show()


In [ ]:
# SOC分布
if os.path.exists(BATTERY_TRACE_FILE):
    battery_trace = pd.read_parquet(BATTERY_TRACE_FILE)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 充电前SOC分布
    axes[0].hist(battery_trace['SOC_before_pct'], bins=30, edgecolor='black', alpha=0.7, color='orange')
    axes[0].set_xlabel('SOC (%)')
    axes[0].set_ylabel('事件数')
    axes[0].set_title('充电前SOC分布')
    axes[0].axvline(20, color='red', linestyle='--', label='20% 低电量线')
    axes[0].legend()

    # 充电后SOC分布
    axes[1].hist(battery_trace['SOC_after_pct'], bins=30, edgecolor='black', alpha=0.7, color='green')
    axes[1].set_xlabel('SOC (%)')
    axes[1].set_ylabel('事件数')
    axes[1].set_title('充电后SOC分布')

    plt.tight_layout()
    plt.show()


In [ ]:
# 两次充电间距离分布
if os.path.exists(ENERGY_GAP_FILE):
    energy_gap_df = pd.read_parquet(ENERGY_GAP_FILE)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 距离分布
    axes[0].hist(energy_gap_df['distance_km'], bins=50, edgecolor='black', alpha=0.7, color='purple')
    axes[0].set_xlabel('距离 (km)')
    axes[0].set_ylabel('事件数')
    axes[0].set_title('两次充电间行驶距离分布')

    # 能耗分布
    axes[1].hist(energy_gap_df['energy_used_kWh'], bins=50, edgecolor='black', alpha=0.7, color='brown')
    axes[1].set_xlabel('能耗 (kWh)')
    axes[1].set_ylabel('事件数')
    axes[1].set_title('两次充电间能耗分布')

    plt.tight_layout()
    plt.show()


In [ ]:
# 充电事件空间分布地图
if 'charging_events' in locals() and not charging_events.empty:
    # 计算地图中心
    center_lat = charging_events['stay_lat'].mean()
    center_lon = charging_events['stay_lon'].mean()

    # 创建地图
    m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='OpenStreetMap')

    # 添加充电站
    station_cluster = MarkerCluster(name='Charging Stations').add_to(m)
    for _, row in stations_df.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=5,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6,
            popup=f"Station {row['station_id']}<br>Piles: {row['num_piles']}"
        ).add_to(station_cluster)

    # 添加充电事件（采样显示，避免过多）
    sample_size = min(1000, len(charging_events))
    charge_sample = charging_events.sample(n=sample_size) if len(charging_events) > sample_size else charging_events

    charge_cluster = MarkerCluster(name='Charging Events').add_to(m)
    for _, row in charge_sample.iterrows():
        folium.CircleMarker(
            location=[row['stay_lat'], row['stay_lon']],
            radius=3,
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.4,
            popup=f"Taxi: {row['taxiid']}<br>Duration: {row['duration_s']/60:.1f} min"
        ).add_to(charge_cluster)

    folium.LayerControl().add_to(m)
    m


## 8. 单车辆案例分析

可以在这里选择特定车辆进行详细分析


In [ ]:
# 选择一个车辆进行详细分析
TID = 'UUUB0C0M7'  # 可以改成任意车辆ID

if 'charging_events' in locals():
    ce_tid = charging_events[charging_events['taxiid'] == TID].sort_values('start_time')

    if not ce_tid.empty:
        print(f"车辆 {TID} 的充电事件:")
        print(f"总充电次数: {len(ce_tid)}")
        print(ce_tid[['start_time', 'end_time', 'duration_s', 'nearest_station_id']])

        if os.path.exists(BATTERY_TRACE_FILE):
            battery_trace_tid = pd.read_parquet(BATTERY_TRACE_FILE)
            battery_trace_tid = battery_trace_tid[battery_trace_tid['taxiid'] == TID].sort_values('start_time')

            if not battery_trace_tid.empty:
                print(f"\nSOC轨迹:")
                print(battery_trace_tid[['start_time', 'SOC_before_pct', 'SOC_after_pct',
                                         'energy_used_kWh', 'energy_in_kWh']])
    else:
        print(f"车辆 {TID} 没有充电事件")
else:
    print("请先加载充电事件数据")


In [ ]:
# ========== 专用站充电事件与充电站空间匹配 ==========
# 功能：将识别出的在专用充电站充电事件，和charge station深圳里面的站进行空间匹配
# 第一步：为每个充电事件找到最近的站
# 第二步：筛选出距离小于300米的站和事件
# 第三步：统计每个站的事件数量

import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

def haversine_np(lon1, lat1, lon2, lat2):
    """
    计算两点之间的Haversine距离（米）
    使用向量化计算，支持数组输入
    """
    R = 6371000  # 地球半径（米）
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# 1. 加载专用站充电事件数据
# 尝试多个可能的文件名
dedicated_file_options = [
    'output_v4/private_gap_charging_events_p.parquet',
    'output_v4/centralized_charging_events_p.parquet'
]
DEDICATED_CHARGES_FILE = None
for file_path in dedicated_file_options:
    if os.path.exists(file_path):
        DEDICATED_CHARGES_FILE = file_path
        break

if DEDICATED_CHARGES_FILE and os.path.exists(DEDICATED_CHARGES_FILE):
    dedicated_charges_df = pd.read_parquet(DEDICATED_CHARGES_FILE)
    logging.info(f"已加载专用站充电事件: {len(dedicated_charges_df)} 条，文件: {DEDICATED_CHARGES_FILE}")
    print(f"专用站充电事件数据列: {dedicated_charges_df.columns.tolist()}")
    print(dedicated_charges_df.head())
else:
    logging.error(f"专用站充电事件文件不存在，尝试过的路径: {dedicated_file_options}")
    dedicated_charges_df = pd.DataFrame()

# 2. 加载充电站数据（charge station深圳）
# 尝试多个可能的文件名
station_files = ['chargestation深圳.csv', 'charge station深圳.csv']
stations_df = None
station_file_used = None

for station_file in station_files:
    if os.path.exists(station_file):
        # 先尝试读取，不强制要求station_id列
        try:
            stations_df = pd.read_csv(station_file)
        except:
            stations_df = pd.read_csv(station_file, encoding='utf-8-sig')
        station_file_used = station_file
        logging.info(f"已加载充电站数据: {len(stations_df)} 个站，文件: {station_file}")
        break

if stations_df is None:
    # 如果当前目录没有，尝试上级目录
    for station_file in station_files:
        parent_path = f'../{station_file}'
        if os.path.exists(parent_path):
            try:
                stations_df = pd.read_csv(parent_path)
            except:
                stations_df = pd.read_csv(parent_path, encoding='utf-8-sig')
            station_file_used = parent_path
            logging.info(f"已加载充电站数据: {len(stations_df)} 个站，文件: {parent_path}")
            break

if stations_df is None:
    logging.error("未找到充电站数据文件，请确认文件路径")
    stations_df = pd.DataFrame()
else:
    # 如果没有station_id列，创建一个（使用索引）
    if 'station_id' not in stations_df.columns:
        stations_df['station_id'] = range(len(stations_df))
        logging.info("充电站数据没有station_id列，已自动创建")

# 检查必要的列是否存在
if not dedicated_charges_df.empty and not stations_df.empty:
    # 检查专用站充电事件的坐标列名（可能是start_lon/start_lat或stay_lon/stay_lat）
    lon_col = None
    lat_col = None
    for col in dedicated_charges_df.columns:
        if 'lon' in col.lower() and ('start' in col.lower() or 'stay' in col.lower()):
            lon_col = col
        if 'lat' in col.lower() and ('start' in col.lower() or 'stay' in col.lower()):
            lat_col = col

    if lon_col is None or lat_col is None:
        logging.error(f"未找到坐标列，可用列: {dedicated_charges_df.columns.tolist()}")
    else:
        print(f"\n使用坐标列: {lon_col}, {lat_col}")

        # 检查充电站数据的坐标列名（支持中英文）
        station_lon_col = None
        station_lat_col = None

        # 优先查找中文列名
        for col in stations_df.columns:
            if col == '经度' or '经度' in col:
                station_lon_col = col
            if col == '纬度' or '纬度' in col:
                station_lat_col = col

        # 如果没找到中文列名，查找英文列名
        if station_lon_col is None or station_lat_col is None:
            for col in stations_df.columns:
                col_lower = col.lower()
                if 'longitude' in col_lower or ('lon' in col_lower and 'longitude' not in col_lower):
                    if station_lon_col is None:
                        station_lon_col = col
                if 'latitude' in col_lower or ('lat' in col_lower and 'latitude' not in col_lower):
                    if station_lat_col is None:
                        station_lat_col = col

        if station_lon_col is None or station_lat_col is None:
            logging.error(f"未找到充电站坐标列，可用列: {stations_df.columns.tolist()}")
            print(f"请检查充电站数据是否包含'经度'/'纬度'或'longitude'/'latitude'列")
        else:
            print(f"使用充电站坐标列: {station_lon_col}, {station_lat_col}")

            # 第一步：为每个充电事件找到最近的站
            print("\n========== 第一步：为每个充电事件找到最近的站 ==========")

            # 构建充电站KDTree
            station_coords = stations_df[[station_lon_col, station_lat_col]].values
            station_tree = cKDTree(station_coords)
            station_id_lookup = stations_df['station_id'].values

            # 获取充电事件的坐标
            event_coords = dedicated_charges_df[[lon_col, lat_col]].values

            # 查询最近的站（返回距离和索引）
            # 注意：cKDTree返回的是欧氏距离（度），需要转换为米
            distances_deg, indices = station_tree.query(event_coords, k=1)

            # 获取对应的站ID和站坐标
            nearest_station_ids = station_id_lookup[indices]
            nearest_station_coords = station_coords[indices]

            # 使用Haversine公式计算实际距离（米）
            event_lons = dedicated_charges_df[lon_col].values
            event_lats = dedicated_charges_df[lat_col].values
            station_lons = nearest_station_coords[:, 0]
            station_lats = nearest_station_coords[:, 1]

            distances_m = haversine_np(event_lons, event_lats, station_lons, station_lats)

            # 添加到充电事件数据中
            dedicated_charges_df['nearest_station_id'] = nearest_station_ids
            dedicated_charges_df['distance_to_station_m'] = distances_m

            print(f"完成最近站匹配，共处理 {len(dedicated_charges_df)} 个充电事件")
            print(f"最近距离统计:")
            print(f"  平均距离: {dedicated_charges_df['distance_to_station_m'].mean():.2f} 米")
            print(f"  中位数距离: {dedicated_charges_df['distance_to_station_m'].median():.2f} 米")
            print(f"  最小距离: {dedicated_charges_df['distance_to_station_m'].min():.2f} 米")
            print(f"  最大距离: {dedicated_charges_df['distance_to_station_m'].max():.2f} 米")

            # 第二步：筛选出距离小于300米的站和事件
            print("\n========== 第二步：筛选出距离小于300米的站和事件 ==========")
            DISTANCE_THRESHOLD = 500  # 300米

            matched_charges_df = dedicated_charges_df[
                dedicated_charges_df['distance_to_station_m'] < DISTANCE_THRESHOLD
            ].copy()

            print(f"筛选前: {len(dedicated_charges_df)} 个充电事件")
            print(f"筛选后: {len(matched_charges_df)} 个充电事件（距离 < {DISTANCE_THRESHOLD}米）")
            print(f"匹配率: {len(matched_charges_df) / len(dedicated_charges_df) * 100:.2f}%")

            # 显示匹配结果示例
            print("\n匹配结果示例（前10条）:")
            print(matched_charges_df[['taxiid', 'nearest_station_id', 'distance_to_station_m',
                                     lon_col, lat_col, 'start_time', 'end_time']].head(10))

            # 第三步：统计每个站的事件数量
            print("\n========== 第三步：统计每个站的事件数量 ==========")

            # 重新计算事件数量和涉及车辆数
            station_stats = matched_charges_df.groupby('nearest_station_id').agg({
                'taxiid': ['count', 'nunique'],
                'distance_to_station_m': ['mean', 'min', 'max']
            }).reset_index()
            station_stats.columns = ['station_id', 'event_count', 'vehicle_count',
                                    'avg_distance_m', 'min_distance_m', 'max_distance_m']

            # 合并充电站信息
            station_stats = station_stats.merge(
                stations_df[['station_id', station_lon_col, station_lat_col]],
                on='station_id',
                how='left'
            )

            # 按事件数量排序
            station_stats = station_stats.sort_values('event_count', ascending=False).reset_index(drop=True)

            print(f"\n共匹配到 {len(station_stats)} 个充电站")
            print(f"\n事件数量最多的前20个站:")
            print(station_stats.head(20).to_string(index=False))

            print(f"\n统计摘要:")
            print(f"  总匹配事件数: {station_stats['event_count'].sum()}")
            print(f"  平均每站事件数: {station_stats['event_count'].mean():.2f}")
            print(f"  中位数每站事件数: {station_stats['event_count'].median():.2f}")
            print(f"  最多事件数: {station_stats['event_count'].max()}")
            print(f"  最少事件数: {station_stats['event_count'].min()}")

            # 保存结果
            output_file = 'output_v4/dedicated_charges_matched_stations.parquet'
            matched_charges_df.to_parquet(output_file, index=False)
            logging.info(f"匹配结果已保存到: {output_file}")

            stats_file = 'output_v4/station_event_counts.csv'
            station_stats.to_csv(stats_file, index=False, encoding='utf-8-sig')
            logging.info(f"站点事件统计已保存到: {stats_file}")

            print(f"\n结果已保存:")
            print(f"  - 匹配的充电事件: {output_file}")
            print(f"  - 站点事件统计: {stats_file}")
else:
    print("数据加载失败，请检查文件路径")

In [ ]:
stations_df.head(10)

In [ ]:
# 统计每个站的事件数量并计算分位数
counts = matched_charges_df.groupby('nearest_station_id').size()
print(counts.quantile([0.05, 0.25,0.35, 0.5, 0.75, 0.95]))

In [ ]:
matched_charges_df['taxiid'].nunique()

In [ ]:
import folium

# 获取匹配站的信息（包含坐标和事件数量）
station_stats = matched_charges_df.groupby('nearest_station_id').agg({
    'taxiid': 'count',
    'distance_to_station_m': 'mean'
}).reset_index()
station_stats.columns = ['station_id', 'event_count', 'avg_distance']

# 合并充电站坐标信息
station_stats = station_stats.merge(
    stations_df[['station_id', '经度', '纬度']],
    on='station_id',
    how='left'
)

# 创建地图（深圳中心）
m = folium.Map(location=[22.5431, 114.0579], zoom_start=11)

# 添加标记点
for _, row in station_stats.iterrows():
    folium.CircleMarker(
        location=[row['纬度'], row['经度']],
        radius=5,
        popup=f"站{row['station_id']}: {row['event_count']}个事件",
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.6
    ).add_to(m)

m

In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
from scipy.spatial import cKDTree
import gc

def haversine_np(lon1, lat1, lon2, lat2):
    R = 6371000
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

dedicated_file = 'output_v4/centralized_charging_events_p.parquet'
dedicated_charges_df = pd.read_parquet(dedicated_file)
dedicated_charges_df = dedicated_charges_df[dedicated_charges_df['taxiid'].isin(filterids)].copy()

# 分批聚类
batch_size = 10000
all_cluster_centers = []
eps_deg =500 / 6371000
total_noise_points = 0
total_clustered_points = 0

n_batches = (len(dedicated_charges_df) + batch_size - 1) // batch_size
print(f"数据量: {len(dedicated_charges_df)}，分{n_batches}批处理...")
print(f"每批大小: {batch_size}")

for i in range(n_batches):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(dedicated_charges_df))
    batch_df = dedicated_charges_df.iloc[start_idx:end_idx].copy()

    # 关键修复：交换坐标顺序，改为 [lat, lon] 格式
    coords = batch_df[['start_lat', 'start_lon']].values  # 注意：lat在前，lon在后
    coords_rad = np.radians(coords)

    dbscan = DBSCAN(
        eps=eps_deg,
        min_samples=2,
        metric='haversine',
        algorithm='ball_tree'
    )

    try:
        labels = dbscan.fit_predict(coords_rad)

        # 统计噪声点
        noise_count = np.sum(labels == -1)
        clustered_count = np.sum(labels != -1)
        total_noise_points += noise_count
        total_clustered_points += clustered_count

        # 收集聚类中心（注意：coords已经是[lat, lon]格式，所以取mean时顺序不变）
        unique_labels = set(labels)
        for cluster_id in unique_labels:
            if cluster_id != -1:
                cluster_mask = labels == cluster_id
                cluster_points = batch_df[cluster_mask]
                # 注意：这里仍然用start_lon和start_lat，因为DataFrame列名没变
                all_cluster_centers.append([
                    cluster_points['start_lon'].mean(),  # 经度
                    cluster_points['start_lat'].mean()   # 纬度
                ])

        del coords, coords_rad, labels, batch_df
        gc.collect()

    except MemoryError:
        print(f"  警告: 第{i+1}批内存不足，跳过")
        continue
    except Exception as e:
        print(f"  警告: 第{i+1}批处理出错: {e}")
        continue

    if (i + 1) % 5 == 0 or (i + 1) == n_batches:
        print(f"  已完成 {i+1}/{n_batches} 批，当前聚类中心数: {len(all_cluster_centers)}，累计噪声点: {total_noise_points}")

print(f"\n分批聚类完成！")
print(f"  总数据点数: {len(dedicated_charges_df)}")
print(f"  聚类成功的点数: {total_clustered_points}")
print(f"  被剔除的噪声点数: {total_noise_points}")
print(f"  噪声点比例: {total_noise_points / len(dedicated_charges_df) * 100:.2f}%")
print(f"  生成的聚类中心数: {len(all_cluster_centers)}")



In [ ]:
# 合并相近的聚类中心（去重，200米内合并）
print(f"\n合并前聚类中心数: {len(all_cluster_centers)}")

if len(all_cluster_centers) > 0:
    centers_df = pd.DataFrame(all_cluster_centers, columns=['center_lon', 'center_lat'])
    # 合并时也要注意坐标顺序
    centers_coords = centers_df[['center_lat', 'center_lon']].values  # [lat, lon]格式
    centers_rad = np.radians(centers_coords)

    dbscan_centers = DBSCAN(
        eps=eps_deg*0.4,
        min_samples=1,
        metric='haversine',
        algorithm='ball_tree'
    )
    center_labels = dbscan_centers.fit_predict(centers_rad)

    final_centers = []
    for cluster_id in set(center_labels):
        if cluster_id != -1:
            cluster_centers = centers_df[center_labels == cluster_id]
            final_centers.append([
                cluster_centers['center_lon'].mean(),
                cluster_centers['center_lat'].mean()
            ])

    cluster_centers_df = pd.DataFrame(final_centers, columns=['center_lon', 'center_lat'])
    print(f"合并后聚类中心数: {len(cluster_centers_df)}")
    print(f"合并掉的重复中心数: {len(all_cluster_centers) - len(cluster_centers_df)}")

    # 验证坐标范围（应该在深圳范围内）
    print(f"\n坐标范围验证:")
    print(f"  经度范围: {cluster_centers_df['center_lon'].min():.4f} ~ {cluster_centers_df['center_lon'].max():.4f}")
    print(f"  纬度范围: {cluster_centers_df['center_lat'].min():.4f} ~ {cluster_centers_df['center_lat'].max():.4f}")
    print(f"  深圳范围: 经度 113.7-114.6, 纬度 22.4-22.9")
else:
    cluster_centers_df = pd.DataFrame(columns=['center_lon', 'center_lat'])
    print("警告: 没有聚类中心")

In [72]:
station_files = ['chargestation深圳.csv', 'charge station深圳.csv']
stations_df = None
station_file_used = None

for station_file in station_files:
    if os.path.exists(station_file):
        # 先尝试读取，不强制要求station_id列
        try:
            stations_df = pd.read_csv(station_file)
        except:
            stations_df = pd.read_csv(station_file, encoding='utf-8-sig')
        station_file_used = station_file
        logging.info(f"已加载充电站数据: {len(stations_df)} 个站，文件: {station_file}")
        break

if stations_df is None:
    # 如果当前目录没有，尝试上级目录
    for station_file in station_files:
        parent_path = f'../{station_file}'
        if os.path.exists(parent_path):
            try:
                stations_df = pd.read_csv(parent_path)
            except:
                stations_df = pd.read_csv(parent_path, encoding='utf-8-sig')
            station_file_used = parent_path
            logging.info(f"已加载充电站数据: {len(stations_df)} 个站，文件: {parent_path}")
            break

if stations_df is None:
    logging.error("未找到充电站数据文件，请确认文件路径")
    stations_df = pd.DataFrame()
else:
    # 如果没有station_id列，创建一个（使用索引）
    if 'station_id' not in stations_df.columns:
        stations_df['station_id'] = range(len(stations_df))
        logging.info("充电站数据没有station_id列，已自动创建")

2025-11-25 13:45:08,291 | INFO | 已加载充电站数据: 4423 个站，文件: chargestation深圳.csv
2025-11-25 13:45:08,293 | INFO | 充电站数据没有station_id列，已自动创建


In [ ]:
from scipy.spatial import cKDTree
import numpy as np

def haversine_np(lon1, lat1, lon2, lat2):
    R = 6371000
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# 准备充电站坐标
station_coords = stations_df[['经度', '纬度']].values
station_tree = cKDTree(station_coords)
station_id_lookup = stations_df['station_id'].values

# 准备聚类中心坐标
cluster_coords = cluster_centers_df[['center_lon', 'center_lat']].values

# 查找每个聚类中心最近的充电站
distances_deg, indices = station_tree.query(cluster_coords, k=1)
nearest_station_ids = station_id_lookup[indices]
nearest_station_coords = station_coords[indices]

# 计算实际距离（米）
distances_m = haversine_np(
    cluster_centers_df['center_lon'].values,
    cluster_centers_df['center_lat'].values,
    nearest_station_coords[:, 0],
    nearest_station_coords[:, 1]
)

# 添加到DataFrame
cluster_centers_df['nearest_station_id'] = nearest_station_ids
cluster_centers_df['distance_to_station_m'] = distances_m

# 统计距离
print("=" * 60)
print("聚类中心与充电站匹配统计")
print("=" * 60)
print(f"总聚类中心数: {len(cluster_centers_df)}")
print(f"\n距离统计（米）:")
print(f"  最小距离: {distances_m.min():.2f}")
print(f"  最大距离: {distances_m.max():.2f}")
print(f"  平均距离: {distances_m.mean():.2f}")
print(f"  中位数距离: {np.median(distances_m):.2f}")
print(f"  标准差: {distances_m.std():.2f}")
print(f"\n距离分布:")
print(f"  < 100米: {np.sum(distances_m < 100)} ({np.sum(distances_m < 100)/len(distances_m)*100:.2f}%)")
print(f"  < 200米: {np.sum(distances_m < 200)} ({np.sum(distances_m < 200)/len(distances_m)*100:.2f}%)")
print(f"  < 500米: {np.sum(distances_m < 500)} ({np.sum(distances_m < 500)/len(distances_m)*100:.2f}%)")
print(f"  < 1000米: {np.sum(distances_m < 1000)} ({np.sum(distances_m < 1000)/len(distances_m)*100:.2f}%)")
print(f"  >= 1000米: {np.sum(distances_m >= 1000)} ({np.sum(distances_m >= 1000)/len(distances_m)*100:.2f}%)")

# 显示前10个匹配结果
print(f"\n前10个匹配结果:")
print(cluster_centers_df[['center_lon', 'center_lat', 'nearest_station_id', 'distance_to_station_m']].head(10))

In [ ]:
import folium
from folium import plugins

# 创建深圳地图（中心点设为深圳市中心）
m = folium.Map(
    location=[22.5431, 114.0579],  # 深圳中心坐标
    zoom_start=11,
    tiles='OpenStreetMap'
)

# 添加聚类中心点（红色）
for idx, row in cluster_centers_df.iterrows():
    folium.CircleMarker(
        location=[row['center_lat'], row['center_lon']],
        radius=5,
        popup=f"聚类中心 {idx}<br>最近站: {row['nearest_station_id']}<br>距离: {row['distance_to_station_m']:.1f}米",
        tooltip=f"聚类中心 {idx}",
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.7,
        weight=2
    ).add_to(m)

# 添加匹配的充电站（蓝色）
matched_stations = cluster_centers_df['nearest_station_id'].unique()
for station_id in matched_stations:
    station_info = stations_df[stations_df['station_id'] == station_id].iloc[0]
    folium.CircleMarker(
        location=[station_info['纬度'], station_info['经度']],
        radius=8,
        popup=f"充电站 {station_id}",
        tooltip=f"充电站 {station_id}",
        color='blue',
        fill=True,
        fillColor='blue',
        fillOpacity=0.7,
        weight=2
    ).add_to(m)

# 添加连线（从聚类中心到最近的充电站）
for idx, row in cluster_centers_df.iterrows():
    station_info = stations_df[stations_df['station_id'] == row['nearest_station_id']].iloc[0]
    folium.PolyLine(
        locations=[
            [row['center_lat'], row['center_lon']],
            [station_info['纬度'], station_info['经度']]
        ],
        color='gray',
        weight=1,
        opacity=0.5,
        popup=f"距离: {row['distance_to_station_m']:.1f}米"
    ).add_to(m)

# 添加图例
legend_html = '''
<div style="position: fixed;
     bottom: 50px; left: 50px; width: 200px; height: 90px;
     background-color: white; border:2px solid grey; z-index:9999;
     font-size:14px; padding: 10px">
     <p><i class="fa fa-circle" style="color:red"></i> 聚类中心</p>
     <p><i class="fa fa-circle" style="color:blue"></i> 充电站</p>
     <p><i class="fa fa-minus" style="color:gray"></i> 匹配连线</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# 显示地图
print(f"共显示 {len(cluster_centers_df)} 个聚类中心和 {len(matched_stations)} 个充电站")
m

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

def haversine_np(lon1, lat1, lon2, lat2):
    R = 6371000
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# -------- 1. 按深圳范围划分聚类中心 --------
SHENZHEN_BBOX = {
    'lon_min': 113.7,
    'lon_max': 114.7,
    'lat_min': 22.45,
    'lat_max': 22.85
}

in_sz_mask = (
    cluster_centers_df['center_lon'].between(SHENZHEN_BBOX['lon_min'], SHENZHEN_BBOX['lon_max']) &
    cluster_centers_df['center_lat'].between(SHENZHEN_BBOX['lat_min'], SHENZHEN_BBOX['lat_max'])
)

sz_centers = cluster_centers_df[in_sz_mask].copy()
non_sz_centers = cluster_centers_df[~in_sz_mask].copy()

print(f"深圳范围内聚类中心: {len(sz_centers)}")
print(f"深圳范围外聚类中心: {len(non_sz_centers)}")

# -------- 2. 深圳范围内：匹配充电站并计算距离 --------
if not sz_centers.empty:
    station_coords = stations_df[['经度', '纬度']].to_numpy()
    station_tree = cKDTree(station_coords)
    station_ids = stations_df['station_id'].to_numpy()

    query_coords = sz_centers[['center_lon', 'center_lat']].to_numpy()
    _, nearest_idx = station_tree.query(query_coords, k=1)
    nearest_station_ids = station_ids[nearest_idx]
    nearest_station_coords = station_coords[nearest_idx]

    distances_m = haversine_np(
        sz_centers['center_lon'].to_numpy(),
        sz_centers['center_lat'].to_numpy(),
        nearest_station_coords[:, 0],
        nearest_station_coords[:, 1]
    )

    sz_centers['nearest_station_id'] = nearest_station_ids
    sz_centers['distance_to_station_m'] = distances_m

    print("\n深圳范围内距离统计（米）:")
    print(f"  最小: {distances_m.min():.2f}")
    print(f"  中位数: {np.median(distances_m):.2f}")
    print(f"  平均: {distances_m.mean():.2f}")
    print(f"  最大: {distances_m.max():.2f}")
    print(f"  <100m: {np.mean(distances_m < 100)*100:.2f}%")
    print(f"  <200m: {np.mean(distances_m < 200)*100:.2f}%")
    print(f"  <500m: {np.mean(distances_m < 500)*100:.2f}%")
    print(f"  <1000m: {np.mean(distances_m < 1000)*100:.2f}%")
    display(sz_centers[['center_lon','center_lat','nearest_station_id','distance_to_station_m']].head())
else:
    print("\n深圳范围内没有聚类中心。")

# -------- 3. 深圳范围外：统计每个聚类中心对应的事件数 --------
if not non_sz_centers.empty:
    if 'event_count' not in non_sz_centers.columns:
        print("\n警告: cluster_centers_df 缺少 event_count 列，临时设为 1。")
        non_sz_centers = non_sz_centers.assign(event_count=1)

    print(f"\n深圳范围外聚类中心总事件数: {non_sz_centers['event_count'].sum()}")
    print("示例（按事件数排序）:")
    display(
        non_sz_centers[['center_lon','center_lat','event_count']]
        .sort_values('event_count', ascending=False)
        .head(20)
    )
else:
    print("\n深圳范围外没有聚类中心。")

In [ ]:
sz_centers.head()

In [ ]:
import folium
from folium.plugins import MarkerCluster

if len(cluster_centers_df) == 0:
    raise ValueError("cluster_centers_df 为空，请先运行聚类。")

# -------- 1. 计算深圳范围内 / 外集合（沿用上一段代码的结果） --------
SHENZHEN_BBOX = {
    'lon_min': 113.7,
    'lon_max': 114.7,
    'lat_min': 22.45,
    'lat_max': 22.85
}

in_sz_mask = (
    cluster_centers_df['center_lon'].between(SHENZHEN_BBOX['lon_min'], SHENZHEN_BBOX['lon_max']) &
    cluster_centers_df['center_lat'].between(SHENZHEN_BBOX['lat_min'], SHENZHEN_BBOX['lat_max'])
)

sz_centers = cluster_centers_df[in_sz_mask].copy()
non_sz_centers = cluster_centers_df[~in_sz_mask].copy()

if 'event_count' not in cluster_centers_df.columns:
    non_sz_centers = non_sz_centers.assign(event_count=1)

print(f"深圳范围内聚类中心: {len(sz_centers)}")
print(f"深圳范围外聚类中心: {len(non_sz_centers)}")

# -------- 2. 初始化深圳地图 --------
m = folium.Map(location=[22.5431, 114.0579], zoom_start=11, tiles='OpenStreetMap')

# 深圳范围外底图（灰色矩形，方便直观区分）
folium.Rectangle(
    bounds=[
        [SHENZHEN_BBOX['lat_min'], SHENZHEN_BBOX['lon_min']],
        [SHENZHEN_BBOX['lat_max'], SHENZHEN_BBOX['lon_max']]
    ],
    color='green',
    weight=2,
    fill=False,
    tooltip='Shenzhen bounding box'
).add_to(m)

# -------- 3. 深圳范围内聚类中心 --------
if not sz_centers.empty:
    sz_cluster = MarkerCluster(name='深圳内聚类中心').add_to(m)
    for _, row in sz_centers.iterrows():
        popup_lines = [
            f"lon: {row['center_lon']:.5f}",
            f"lat: {row['center_lat']:.5f}"
        ]
        if 'nearest_station_id' in row:
            popup_lines.append(f"nearest station: {row['nearest_station_id']}")
        if 'distance_to_station_m' in row:
            popup_lines.append(f"distance: {row['distance_to_station_m']:.1f} m")
        folium.CircleMarker(
            location=[row['center_lat'], row['center_lon']],
            radius=4,
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.7,
            weight=1,
            popup='<br>'.join(popup_lines),
            tooltip='SZ center'
        ).add_to(sz_cluster)
else:
    folium.map.Marker(
        [22.6, 114.3],
        icon=folium.DivIcon(html="<b style='color:red;'>No SZ centers</b>")
    ).add_to(m)

# -------- 4. 深圳范围外聚类中心 --------
if not non_sz_centers.empty:
    outside_layer = folium.FeatureGroup(name='深圳外聚类中心').add_to(m)
    for _, row in non_sz_centers.iterrows():
        size = max(3, min(row['event_count'], 30))  # 用事件数量控制圆圈大小（限制在3~30）
        folium.CircleMarker(
            location=[row['center_lat'], row['center_lon']],
            radius=size / 2,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.5,
            weight=1,
            popup=(
                f"lon: {row['center_lon']:.5f}<br>"
                f"lat: {row['center_lat']:.5f}<br>"
                f"event_count: {row['event_count']}"
            ),
            tooltip=f"外部中心（{row['event_count']} events）"
        ).add_to(outside_layer)
else:
    folium.map.Marker(
        [22.2, 113.1],
        icon=folium.DivIcon(html="<b style='color:blue;'>No outside centers</b>")
    ).add_to(m)

# -------- 5. 图层控制 + 图例 --------
folium.LayerControl().add_to(m)

legend_html = '''
<div style="
    position: fixed;
    bottom: 40px; left: 40px;
    width: 220px; height: 140px;
    background-color: white;
    border: 2px solid grey;
    z-index: 9999;
    font-size: 14px;
    padding: 10px;">
    <h4 style="margin-top:0;">图例</h4>
    <p style="margin:0;"><span style="color:red;">●</span> 深圳内聚类中心</p>
    <p style="margin:0;"><span style="color:blue;">●</span> 深圳外聚类中心（大小 ~ 事件数）</p>
    <p style="margin:0;">■ 绿色矩形：深圳范围</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

m

In [73]:
import numpy as np
from scipy.spatial import cKDTree

# --- 参数 ---
SHENZHEN_BBOX = dict(lon_min=113.7, lon_max=114.7, lat_min=22.45, lat_max=22.85)
DIST_THRESHOLD_M = 500

# --- 1. 数据准备 ---
# 专用站充电事件（确保已经加载到 dedicated_charges_df）
events = dedicated_charges_df.copy()

# 充电站信息（确保已经加载到 stations_df）
station_id_col = 'station_id'
# 自动找经纬度列
lon_col = next(col for col in stations_df.columns if any(k in col for k in ['经度', 'longitude', 'lon']))
lat_col = next(col for col in stations_df.columns if any(k in col for k in ['纬度', 'latitude', 'lat']))
# 尽力找到名称 / 类型列（可选）
name_col = next((col for col in stations_df.columns if any(k in col.lower() for k in ['name', 'station', '站名'])), None)
category_col = next((col for col in stations_df.columns if any(k in col for k in ['类型', '分类', 'category'])), None)

# --- 2. 只保留深圳范围内的专用站充电事件 ---
in_sz_mask = (
    events['start_lon'].between(SHENZHEN_BBOX['lon_min'], SHENZHEN_BBOX['lon_max']) &
    events['start_lat'].between(SHENZHEN_BBOX['lat_min'], SHENZHEN_BBOX['lat_max'])
)
events_sz = events.loc[in_sz_mask].copy()
print(f"专用站充电事件（深圳范围）: {len(events_sz):,}")

if events_sz.empty:
    raise ValueError("深圳范围内没有专用站充电事件，检查数据或经纬度列名。")

# --- 3. 最近充电站匹配 + 距离 ---
station_coords = stations_df[[lon_col, lat_col]].to_numpy()
station_tree = cKDTree(station_coords)
station_ids = stations_df[station_id_col].to_numpy()

event_coords = events_sz[['start_lon', 'start_lat']].to_numpy()
_, nearest_idx = station_tree.query(event_coords, k=1)
nearest_station_ids = station_ids[nearest_idx]
nearest_station_coords = station_coords[nearest_idx]

def haversine_np(lon1, lat1, lon2, lat2):
    R = 6371000
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

distances_m = haversine_np(
    events_sz['start_lon'].to_numpy(),
    events_sz['start_lat'].to_numpy(),
    nearest_station_coords[:, 0],
    nearest_station_coords[:, 1]
)

events_sz['nearest_station_id'] = nearest_station_ids
events_sz['distance_to_station_m'] = distances_m
events_sz['is_charging'] = events_sz['distance_to_station_m'] <= DIST_THRESHOLD_M

# --- 4. 距离统计 ---
print("\n距离统计（深圳范围专用站事件）：")
print(f"  中位数: {np.median(distances_m):.1f} m")
print(f"  平均: {np.mean(distances_m):.1f} m")
print(f"  <=100m: {np.mean(distances_m <= 100)*100:.2f}%")
print(f"  <=200m: {np.mean(distances_m <= 200)*100:.2f}%")
print(f"  <=500m: {np.mean(events_sz['is_charging'])*100:.2f}%")
# --- 5. 统计真正去充电的事件（<=500m） ---
real_dedi_charging_events = events_sz.loc[events_sz['is_charging']].copy()
print(f"\n判定为“去充电”的事件数: {len(real_dedi_charging_events):} / {len(events_sz):}")



专用站充电事件（深圳范围）: 316,576

距离统计（深圳范围专用站事件）：
  中位数: 189.5 m
  平均: 247.9 m
  <=100m: 18.95%
  <=200m: 52.50%
  <=500m: 90.17%

判定为“去充电”的事件数: 285464 / 316576


In [74]:
real_dedi_charging_events.head(5)

,taxiid,start_time,end_time,duration_s,start_lon,start_lat,charge_type,nearest_station_id,distance_to_station_m,is_charging
0,UUUB0C0M7,2020-01-01 04:17:29,2020-01-01 05:48:13,5444.0,114.143974,22.557758,dedicated_station,1801,177.351840,True
1,UUUB0C0M7,2020-01-01 16:18:31,2020-01-01 17:47:47,5356.0,114.143753,22.557638,dedicated_station,1801,152.227049,True
2,UUUB0C0M7,2020-01-02 04:16:24,2020-01-02 05:41:39,5115.0,114.143974,22.557781,dedicated_station,1801,177.993366,True
3,UUUB0C0M7,2020-01-02 16:09:34,2020-01-02 17:32:36,4982.0,114.143990,22.557774,dedicated_station,1801,179.291552,True
4,UUUB0C0M7,2020-01-03 04:33:40,2020-01-03 05:44:59,4279.0,114.143890,22.557579,dedicated_station,1801,165.006212,True


In [ ]:
# # ========== 合并公共充电事件和专用站充电事件，用于后续能耗计算 ==========
#
#
#
# # 3. 格式化公共充电事件（统一字段）
# if not public_charges_df.empty:
#     public_charges_formatted = pd.DataFrame({
#         'taxiid': public_charges_df['taxiid'],
#         'nearest_station_id': public_charges_df['nearest_station_id'],
#         'start_time': public_charges_df['start_time'],
#         'end_time': public_charges_df['end_time'],
#         'duration_s': public_charges_df['duration_s'],
#         'stay_lon': public_charges_df['stay_lon'],
#         'stay_lat': public_charges_df['stay_lat'],
#         'charge_type': 'public'
#     })
# else:
#     public_charges_formatted = pd.DataFrame()
#
# # 4. 格式化专用站充电事件（统一字段）
# if not real_dedi_charging_events.empty:
#     dedicated_charges_formatted = pd.DataFrame({
#         'taxiid': dedicated_charges_df['taxiid'],
#         'nearest_station_id': -1,  # 专用站充电标记为-1
#         'start_time': dedicated_charges_df['start_time'],
#         'end_time': dedicated_charges_df['end_time'],
#         'duration_s': dedicated_charges_df['duration_s'],
#         'stay_lon': dedicated_charges_df['start_lon'],
#         'stay_lat': dedicated_charges_df['start_lat'],
#         'charge_type': 'dedicated_station'
#     })
# else:
#     dedicated_charges_formatted = pd.DataFrame()
#
# # 5. 合并所有充电事件
# if not public_charges_formatted.empty and not dedicated_charges_formatted.empty:
#     newall_charging_events = pd.concat([
#         public_charges_formatted,
#         dedicated_charges_formatted
#     ], ignore_index=True)
#     logging.info(f"合并成功，总事件数: {len(newall_charging_events)}")
#
# # 6. 按车辆和时间排序
# if not newall_charging_events.empty:
#     newall_charging_events = newall_charging_events.sort_values(['taxiid', 'start_time']).reset_index(drop=True)


In [ ]:
# ========== 基于all_charging_events计算SOC轨迹 ==========

# 检查必要的数据文件是否存在
if all([os.path.exists(CHAR_QUEUE_FILE),
        os.path.exists(ENERGY_GAP_FILE),
        'all_charging_events' in locals() or 'all_charging_events' in globals()]):

    # 1. 加载数据
    cq = pd.read_parquet(CHAR_QUEUE_FILE)  # 排队结果
    eg = pd.read_parquet(ENERGY_GAP_FILE)  # 能耗数据（包含charge_type）
    # cq=cq[cq['taxiid'].isin(seleCTID)]
    # eg=eg[eg['taxiid'].isin(seleCTID)]
    # 使用已有的all_charging_events
    battery_trace = all_charging_events.copy()

    # 2. 分离公共和专用站充电事件
    public_mask = battery_trace['charge_type'] == 'public'
    dedicated_mask = battery_trace['charge_type'] == 'dedicated_station'

    battery_trace_public = battery_trace[public_mask].copy()
    battery_trace_dedicated = battery_trace[dedicated_mask].copy()

    # 3. 处理公共充电事件：合并排队结果和站点功率
    if not battery_trace_public.empty:
        # 合并排队结果
        battery_trace_public = battery_trace_public.merge(
            cq[['taxiid', 'arrive_time', 'charge_dur', 'wait_dur', 'giveup']],
            left_on=['taxiid', 'start_time'],
            right_on=['taxiid', 'arrive_time'],
            how='left'
        )

        # 合并站点平均功率
        battery_trace_public = battery_trace_public.merge(
            stations_df[['station_id', 'avg_power']],
            left_on='nearest_station_id',
            right_on='station_id',
            how='left'
        )
        battery_trace_public = battery_trace_public.drop(columns=['station_id'], errors='ignore')

    # 4. 处理专用站充电事件：设置默认值
    if not battery_trace_dedicated.empty:
        battery_trace_dedicated['arrive_time'] = battery_trace_dedicated['start_time']
        battery_trace_dedicated['charge_dur'] = battery_trace_dedicated['duration_s']  # 全部都是充电时间
        battery_trace_dedicated['wait_dur'] = 0  # 专用站充电没有等待
        battery_trace_dedicated['giveup'] = False  # 专用站充电不放弃
        battery_trace_dedicated['avg_power'] = 43.0  # 专用站充电功率统一为43kW

    # 5. 合并公共和专用站充电事件
    if not battery_trace_public.empty and not battery_trace_dedicated.empty:
        # 统一列名
        common_cols = ['taxiid', 'nearest_station_id', 'start_time', 'end_time',
                       'duration_s', 'stay_lon', 'stay_lat', 'charge_type',
                       'arrive_time', 'charge_dur', 'wait_dur', 'giveup', 'avg_power']

        battery_trace = pd.concat([
            battery_trace_public[common_cols],
            battery_trace_dedicated[common_cols]
        ], ignore_index=True)
    elif not battery_trace_public.empty:
        battery_trace = battery_trace_public
    elif not battery_trace_dedicated.empty:
        battery_trace = battery_trace_dedicated
    else:
        battery_trace = pd.DataFrame()

    if battery_trace.empty:
        logging.warning("没有充电事件数据，无法计算SOC轨迹")
    else:
        # 6. 按车辆和时间排序
        battery_trace = battery_trace.sort_values(['taxiid', 'start_time']).reset_index(drop=True)
        battery_trace['prev_end'] = battery_trace.groupby('taxiid')['end_time'].shift()

        # 7. 合并能耗数据（两次充电之间的能耗，包含charge_type）
        battery_trace = battery_trace.merge(
            eg[['taxiid', 'charge_start', 'energy_used_kWh', 'missing_s',
                'gap_s', 'drive_s', 'idle_s', 'distance_km', 'charge_type']],
            left_on=['taxiid', 'start_time'],
            right_on=['taxiid', 'charge_start'],
            how='left',
            suffixes=('', '_from_energy')
        )

        # 使用能耗表中的charge_type（更准确，因为它标记的是当前充电事件的类型）
        battery_trace['charge_type'] = battery_trace['charge_type_from_energy'].fillna(battery_trace['charge_type'])
        battery_trace = battery_trace.drop(columns=['charge_type_from_energy', 'charge_start'], errors='ignore')

        # 8. 初始化能耗数据
        battery_trace['energy_used_kWh'] = battery_trace['energy_used_kWh'].fillna(0)


        # 9. 逐车递推SOC（先计算SOC_before，再计算充入电量）
        def calculate_soc_trace_v2(group):
            soc_before_list = []
            soc_after_list = []
            soc_prev = CAP_KWH  # 初始满电

            for _, row in group.iterrows():
                # 计算充电前SOC
                soc_before = max(0, soc_prev - row['energy_used_kWh'])
                soc_before_list.append(soc_before)

                # 计算充入电量（基于充电前SOC）
                charge_dur_hours = row['charge_dur'] / 3600

                # 公共充电：放弃充电的功率为0，否则使用站平均功率
                # 专用站充电：功率统一为43kW
                if row['giveup'] == True:
                    power_kw = 0
                else:
                    power_kw = row['avg_power'] if pd.notna(row['avg_power']) else 0

                # 计算理论充入电量
                energy_in_theoretical = charge_dur_hours * power_kw * CHG_EFFICIENCY

                # 限制充入电量：不能超过（满电量 - 充电前电量）
                max_energy_in = CAP_KWH - soc_before
                energy_in = min(energy_in_theoretical, max_energy_in)

                # 计算充电后SOC
                soc_after = min(CAP_KWH, soc_before + energy_in)
                soc_after_list.append(soc_after)

                soc_prev = soc_after

            group['SOC_before_kWh'] = soc_before_list
            group['SOC_after_kWh'] = soc_after_list
            group['energy_in_kWh'] = [soc_after_list[i] - soc_before_list[i] for i in range(len(soc_before_list))]
            return group


        battery_trace = battery_trace.groupby('taxiid', group_keys=False).apply(calculate_soc_trace_v2)

        # 10. 计算百分比
        battery_trace['SOC_before_pct'] = battery_trace['SOC_before_kWh'] / CAP_KWH * 100
        battery_trace['SOC_after_pct'] = battery_trace['SOC_after_kWh'] / CAP_KWH * 100

        # 11. 选择输出列（包含charge_type）
        output_cols = [
            'taxiid', 'charge_type', 'nearest_station_id', 'start_time', 'end_time',
            'SOC_before_kWh', 'SOC_after_kWh', 'SOC_before_pct', 'SOC_after_pct',
            'energy_used_kWh', 'energy_in_kWh',
            'distance_km', 'drive_s', 'idle_s', 'gap_s', 'missing_s',
            'charge_dur', 'wait_dur', 'giveup'
        ]
        output_cols = [col for col in output_cols if col in battery_trace.columns]

        battery_trace_output = battery_trace[output_cols].copy()
        battery_trace_output.to_parquet(BATTERY_TRACE_FILE, index=False)

        logging.info(f"SOC轨迹计算完成，结果已保存到 {BATTERY_TRACE_FILE}")
        logging.info(f"总充电事件数: {len(battery_trace_output)}")
        logging.info(f"  公共充电: {len(battery_trace_output[battery_trace_output['charge_type'] == 'public'])}")
        logging.info(
            f"  专用站充电: {len(battery_trace_output[battery_trace_output['charge_type'] == 'dedicated_station'])}")

        print("\nSOC轨迹数据示例：")
        print(battery_trace_output.head(10))

        # 12. 统计信息
        print("\n" + "=" * 60)
        print("SOC轨迹统计")
        print("=" * 60)

        public_charges = battery_trace_output[battery_trace_output['charge_type'] == 'public']
        dedicated_charges = battery_trace_output[battery_trace_output['charge_type'] == 'dedicated_station']

        print(f"\n公共充电统计:")
        print(f"  事件数: {len(public_charges)}")
        if len(public_charges) > 0:
            print(f"  平均充入电量: {public_charges['energy_in_kWh'].mean():.2f} kWh")
            print(f"  平均充电前SOC: {public_charges['SOC_before_pct'].mean():.2f}%")
            print(f"  平均充电后SOC: {public_charges['SOC_after_pct'].mean():.2f}%")

        print(f"\n专用站充电统计:")
        print(f"  事件数: {len(dedicated_charges)}")
        if len(dedicated_charges) > 0:
            print(f"  平均充入电量: {dedicated_charges['energy_in_kWh'].mean():.2f} kWh")
            print(f"  平均充电前SOC: {dedicated_charges['SOC_before_pct'].mean():.2f}%")
            print(f"  平均充电后SOC: {dedicated_charges['SOC_after_pct'].mean():.2f}%")

        print(f"\n整体SOC统计:")
        print(f"  充电前SOC平均: {battery_trace_output['SOC_before_pct'].mean():.2f}%")
        print(f"  充电前SOC中位数: {battery_trace_output['SOC_before_pct'].median():.2f}%")
        print(f"  充电前SOC最小值: {battery_trace_output['SOC_before_pct'].min():.2f}%")
        print(f"  充电后SOC平均: {battery_trace_output['SOC_after_pct'].mean():.2f}%")
        print(f"  充电后SOC中位数: {battery_trace_output['SOC_after_pct'].median():.2f}%")
        print(f"  充电后SOC最大值: {battery_trace_output['SOC_after_pct'].max():.2f}%")

        low_soc = (battery_trace_output['SOC_before_pct'] < 20).sum()
        print(f"\n低电量充电次数 (SOC < 20%): {low_soc:,} ({low_soc / len(battery_trace_output) * 100:.2f}%)")

        print("=" * 60)

else:
    logging.warning("缺少必要的数据文件或变量，无法计算SOC轨迹")
    logging.warning(f"需要以下文件:")
    logging.warning(f"  - {CHAR_QUEUE_FILE}")
    logging.warning(f"  - {ENERGY_GAP_FILE}")
    logging.warning(f"需要以下变量:")
    logging.warning(f"  - all_charging_events (合并后的充电事件DataFrame)")
# 假设 battery_trace_output 是你包含SOC轨迹的DataFrame

# 1. 找出所有存在 "充电前SOC < 1%" 记录的车辆ID (taxiid)
#    首先，筛选出所有不符合条件的记录
low_soc_events = battery_trace_output[battery_trace_output['SOC_before_pct'] < 0.1]

#    然后，获取这些记录对应的所有唯一的 taxiid
taxiids_to_exclude = low_soc_events['taxiid'].unique()

# 2. 从原始DataFrame中剔除这些车辆的所有记录
#    使用 .isin() 方法来判断每一行的 'taxiid' 是否在我们要排除的列表中
#    前面的 ~ 符号表示 "取反"，即保留那些 taxiid 不在排除列表中的行
filtered_df = battery_trace_output[~battery_trace_output['taxiid'].isin(taxiids_to_exclude)].copy()

# 3. 打印统计信息，验证筛选结果
original_taxi_count = battery_trace_output['taxiid'].nunique()
filtered_taxi_count = filtered_df['taxiid'].nunique()
removed_taxi_count = original_taxi_count - filtered_taxi_count

print("原始DataFrame信息：")
print(f"  总车辆数: {original_taxi_count}")
print(f"  总事件数: {len(battery_trace_output)}")

print(f"\n需要排除的车辆数: {len(taxiids_to_exclude)}")
print(f"  这些车辆是: {taxiids_to_exclude}")

print("\n筛选后DataFrame信息：")
print(f"  剩余车辆数: {filtered_taxi_count}")
print(f"  剩余事件数: {len(filtered_df)}")

print(f"\n共剔除了 {removed_taxi_count} 辆车的所有数据。")

# 新的DataFrame 'filtered_df' 现在可以用于后续分析
# 可以查看一下筛选后的结果
print("\n筛选后的DataFrame示例：")
print(filtered_df.head())
filterids = filtered_df['taxiid'].unique().tolist()